# code,,v1 try

In [ ]:
#!/usr/bin/env python3
"""
PathSeq vs MiTRI Reads Comparison Pipeline
Compare read intersections between PathSeq and MiTRI and save detailed read information
"""

import os
import sys
import logging
import pandas as pd
import pysam
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, Tuple, List
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing as mp
from datetime import datetime
import re
import gzip

# ==================== Configuration parameters ====================
# CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
CANCERS = ["THCA"]
STATUSES = ["Normal", "Tumor"]

# Path configuration
PATHSEQ_BASE = "/path/to/project/data/cancers_V1"
MITRI_BASE = "/path/to/project/results/cancers_V1_V2.8"
MICROBE_LIST_DIR = "/path/to/project/data/CSVs_20251203"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
OUTPUT_BASE = "/path/to/project/results/summary_V2/reads_compare"

# Parallel configuration
MAX_WORKERS = 40 # 80-core CPU, use half to avoid overload

# Output directory structure
STATISTICS_DIR = "statistics" # statisticstext
DETAILED_READS_DIR = "detailed_reads" # detailedreadsinformation
LOGS_DIR = "logs" # log files


# ==================== Utility functions ====================

def setup_logger(cancer: str, log_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    log_dir.mkdir(parents=True, exist_ok=True)
    log_file = log_dir / f"{cancer}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    logger = logging.getLogger(cancer)
    logger.setLevel(logging.INFO)
    logger.handlers = [] # clear existing handlers
    # file handler
    fh = logging.FileHandler(log_file, encoding='utf-8')
    fh.setLevel(logging.INFO)
    # console handler
    ch = logging.StreamHandler()
    ch.setLevel(logging.INFO)
    # format
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)
    return logger


def normalize_microbe_name(name: str) -> str:
    """textmicrobetext: textandtextfortext"""
    return re.sub(r'[\s/]+', '_', name)


def load_taxid_hierarchy() -> pd.DataFrame:
    """loadtaxidtext"""
    df = pd.read_csv(TAXID_HIERARCHY_FILE, sep='\t')
    return df


def build_microbe_taxid_map(taxid_df: pd.DataFrame) -> Dict[str, Dict]:
    """
    buildmicrobetexttotaxidinformationmapping
    return: {microbe_name: {
    'species_taxid': int,
    'tax_ids': [int, ...], # may have multiple
    'genus_taxid': int
    }}
    """
    microbe_map = defaultdict(lambda: {'species_taxid': None, 'tax_ids': [], 'genus_taxid': None})
    for _, row in taxid_df.iterrows():
        species_name = str(row['species_scientific_name'])
        normalized_name = normalize_microbe_name(species_name)
        species_taxid = int(row['species']) if pd.notna(row['species']) else None
        tax_id = int(row['tax_id']) if pd.notna(row['tax_id']) else None
        genus_taxid = int(row['genus']) if pd.notna(row['genus']) else None
        if normalized_name not in microbe_map or microbe_map[normalized_name]['species_taxid'] is None:
            microbe_map[normalized_name]['species_taxid'] = species_taxid
            microbe_map[normalized_name]['genus_taxid'] = genus_taxid
            if tax_id and tax_id not in microbe_map[normalized_name]['tax_ids']:
                microbe_map[normalized_name]['tax_ids'].append(tax_id)
                return dict(microbe_map)


def load_microbe_list(cancer: str) -> List[str]:
    """loadtextcancer typemicrobe list"""
    csv_file = Path(MICROBE_LIST_DIR) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    # The first column is Taxon, and the Flag column is used for filtering
    if 'Taxon' in df.columns and 'Flag' in df.columns:
        microbes = df[df['Flag'] == 1]['Taxon'].tolist()
    else:
        microbes = df.iloc[:, 0].tolist() # assume the first column is the microbe name
        return [m for m in microbes if m and str(m).strip()]


# ==================== PathSeq Readsextract ====================

def extract_pathseq_reads_pysam(bam_path: Path) -> Dict[Tuple[str, str], Set[int]]:
    """
    textpysamfromPathSeq BAMextractwithYPtextreads
    return: {(read_id, sequence): {taxid1, taxid2, ...}}
    """
    reads_dict = defaultdict(set)
    try:
        with pysam.AlignmentFile(str(bam_path), 'rb') as bam:
            for read in bam:
                # checktextwithYPtext
                if read.has_tag('YP'):
                    yp_value = read.get_tag('YP')
                    # texttaxid(text)
                    taxids = set()
                    for tid in str(yp_value).split(','):
                        tid = tid.strip()
                        if tid.isdigit():
                            taxids.add(int(tid))
                            if taxids:
                                read_id = read.query_name
                                sequence = read.query_sequence
                                if sequence:
                                    reads_dict[(read_id, sequence)].update(taxids)
    except Exception as e:
        logging.error(f"Error reading {bam_path}: {e}")
        return {}
    return dict(reads_dict)


def extract_pathseq_reads_samtools(txt_path: Path) -> Dict[Tuple[str, str], Set[int]]:
    """
    Extract reads with YP tags from the txt file converted by samtools view
    Used to test pysam consistency
    """
    reads_dict = defaultdict(set)
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'): # skipheader
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 11:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                # findYPtext
                yp_value = None
                for field in fields[11:]:
                    if field.startswith('YP:Z:'):
                        yp_value = field[5:]
                        break
                    if yp_value:
                        taxids = set()
                        for tid in yp_value.split(','):
                            tid = tid.strip()
                            if tid.isdigit():
                                taxids.add(int(tid))
                                if taxids:
                                    reads_dict[(read_id, sequence)].update(taxids)
    except Exception as e:
        logging.error(f"Error reading {txt_path}: {e}")
        return {}
    return dict(reads_dict)


# ==================== MiTRI Readsextract ====================

def extract_mitri_reads(txt_path: Path) -> Set[Tuple[str, str]]:
    """
    Extract reads from MiTRI bam.txt files
    return: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'): # skipheader
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 11:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                if sequence and sequence != '*':
                    reads_set.add((read_id, sequence))
    except FileNotFoundError:
        logging.warning(f"File not found: {txt_path}")
        return set()
    except Exception as e:
        logging.error(f"Error reading {txt_path}: {e}")
        return set()
    return reads_set


# ==================== compareanalyze ====================

def compare_reads(mitri_reads: Set[Tuple[str, str]],     pathseq_reads: Dict[Tuple[str, str], Set[int]],
    microbe_taxids: List[int] = None) -> Dict:
"""
Compare MiTRI and PathSeq reads
return:
{
'intersection': set of (read_id, seq),
'mitri_only': set of (read_id, seq),
'intersection_count': int,
'mitri_only_count': int,
'intersection_pct': float,
'mitri_only_pct': float,
'taxid_matched': set (if microbe_taxids is provided)
'pathseq_taxids_for_intersection': dict {(read_id, seq): [taxids]} # new
}
"""
pathseq_reads_keys = set(pathseq_reads.keys())
# textandtext
intersection = mitri_reads & pathseq_reads_keys
mitri_only = mitri_reads - pathseq_reads_keys
total_mitri = len(mitri_reads)
# Save PathSeq taxids for intersecting reads
pathseq_taxids_for_intersection = {}
for read_key in intersection:
    pathseq_taxids_for_intersection[read_key] = list(pathseq_reads[read_key])
    result = {
        'intersection': intersection,
        'mitri_only': mitri_only,
        'intersection_count': len(intersection),
        'mitri_only_count': len(mitri_only),
        'total_mitri': total_mitri,
        'intersection_pct': len(intersection) / total_mitri * 100 if total_mitri > 0 else 0,
        'mitri_only_pct': len(mitri_only) / total_mitri * 100 if total_mitri > 0 else 0,
        'pathseq_taxids_for_intersection': pathseq_taxids_for_intersection
    }
    # textmicrobetaxids, checktaxidmatched
    if microbe_taxids:
        microbe_taxids_set = set(microbe_taxids)
        taxid_matched = set()
        for read_key in intersection:
            pathseq_taxids = pathseq_reads[read_key]
            if pathseq_taxids & microbe_taxids_set:
                taxid_matched.add(read_key)
                result['taxid_matched'] = taxid_matched
                result['taxid_matched_count'] = len(taxid_matched)
                result['taxid_matched_pct'] = len(taxid_matched) / total_mitri * 100 if total_mitri > 0 else 0
                return result


# ==================== Readsinformationsave ====================

def save_reads_to_file(reads_set: Set[Tuple[str, str]],     output_file: Path,
    pathseq_taxids: Dict[Tuple[str, str], List[int]] = None,
    compress: bool = True):
"""
Save read information to file
Args:
reads_set: readstext {(read_id, sequence), ...}
output_file: Output filespath
pathseq_taxids: textreads, textPathSeqtaxidinformation
compress: textgziptext
"""
output_file.parent.mkdir(parents=True, exist_ok=True)
if compress:
    output_file = output_file.with_suffix(output_file.suffix + '.gz')
    open_func = gzip.open
    mode = 'wt'
else:
    open_func = open
    mode = 'w'
    try:
        with open_func(output_file, mode, encoding='utf-8') as f:
            # textheader
            if pathseq_taxids:
                f.write("read_id\tsequence\tpathseq_taxids\n")
                for read_id, sequence in sorted(reads_set):
                    taxids = pathseq_taxids.get((read_id, sequence), [])
                    taxids_str = ','.join(map(str, taxids))
                    f.write(f"{read_id}\t{sequence}\t{taxids_str}\n")
                else:
                    f.write("read_id\tsequence\n")
                    for read_id, sequence in sorted(reads_set):
                        f.write(f"{read_id}\t{sequence}\n")
                        return True
    except Exception as e:
        logging.error(f"Error saving reads to {output_file}: {e}")
        return False


# ==================== sampleprocesstext ====================

def process_sample(args):
    """
    Process all microbe alignments for one sample
    """
    cancer, status, sample, microbes, microbe_taxid_map = args
    logger = logging.getLogger(cancer)
    # PathSeq BAMpath
    pathseq_bam = Path(PATHSEQ_BASE) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.warning(f"PathSeq BAM not found: {pathseq_bam}")
        return None
    # extractPathSeq reads(sampletext, textextract)
    logger.info(f"Processing sample: {cancer}/{status}/{sample}")
    pathseq_reads = extract_pathseq_reads_pysam(pathseq_bam)
    if not pathseq_reads:
        logger.warning(f"No PathSeq reads found for {sample}")
        return None
    logger.info(f" PathSeq reads: {len(pathseq_reads)}")
    # textresults
    sample_results = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'pathseq_reads_count': len(pathseq_reads),
        'microbe_results': [], # Task 1: microbe level
        'sample_level_result': None # Task 2: sample level
    }
    # detailedreadsoutput directory
    sample_reads_dir = Path(OUTPUT_BASE) / DETAILED_READS_DIR / cancer / status / sample
    # Used for Task 2 sample-level aggregation
    all_mitri_reads = set()
    # textmicrobes(text1)
    for microbe in microbes:
        mitri_txt = Path(MITRI_BASE) / cancer / "03.coverage" / status / microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt"
        if not mitri_txt.exists():
            continue
        # Extract MiTRI reads
        mitri_reads = extract_mitri_reads(mitri_txt)
        if not mitri_reads:
            continue
        # Get microbe taxid information
        microbe_info = microbe_taxid_map.get(microbe, {})
        microbe_taxids = microbe_info.get('tax_ids', [])
        # compare(text1andTask 3)
        comparison = compare_reads(mitri_reads, pathseq_reads, microbe_taxids)
        microbe_result = {
            'microbe': microbe,
            'species_taxid': microbe_info.get('species_taxid'),
            'tax_ids': microbe_taxids,
            'genus_taxid': microbe_info.get('genus_taxid'),
            'mitri_reads_count': len(mitri_reads),
            'intersection_count': comparison['intersection_count'],
            'intersection_pct': comparison['intersection_pct'],
            'mitri_only_count': comparison['mitri_only_count'],
            'mitri_only_pct': comparison['mitri_only_pct'],
            'taxid_matched_count': comparison.get('taxid_matched_count', 0),
            'taxid_matched_pct': comparison.get('taxid_matched_pct', 0)
        }
        sample_results['microbe_results'].append(microbe_result)
        # Save detailed reads for Task 1
        microbe_reads_dir = sample_reads_dir / "microbe_level" / microbe
        # Save intersecting reads (with PathSeq taxids)
        if comparison['intersection']:
            intersection_file = microbe_reads_dir / "intersection_reads.txt"
            save_reads_to_file(
                comparison['intersection'],                 intersection_file,
                pathseq_taxids=comparison['pathseq_taxids_for_intersection'],
                compress=True
)
# Save MiTRI-only reads
if comparison['mitri_only']:
    mitri_only_file = microbe_reads_dir / "mitri_only_reads.txt"
    save_reads_to_file(
        comparison['mitri_only'],
        mitri_only_file,
        compress=True
)
# Aggregate to sample level (Task 2)
all_mitri_reads.update(mitri_reads)
logger.info(f" {microbe}: MiTRI={len(mitri_reads)}, "
    f"Intersection={comparison['intersection_count']} ({comparison['intersection_pct']:.2f}%), "
    f"MiTRI_only={comparison['mitri_only_count']} ({comparison['mitri_only_pct']:.2f}%)")
# Task 2: sample levelcompare
if all_mitri_reads:
    sample_comparison = compare_reads(all_mitri_reads, pathseq_reads)
    sample_results['sample_level_result'] = {
        'total_mitri_reads': len(all_mitri_reads),
        'intersection_count': sample_comparison['intersection_count'],
        'intersection_pct': sample_comparison['intersection_pct'],
        'mitri_only_count': sample_comparison['mitri_only_count'],
        'mitri_only_pct': sample_comparison['mitri_only_pct']
    }
    # Save detailed reads for Task 2
    sample_level_dir = sample_reads_dir / "sample_level"
    # Save sample-level intersecting reads (with PathSeq taxids)
    if sample_comparison['intersection']:
        intersection_file = sample_level_dir / "intersection_reads.txt"
        save_reads_to_file(
            sample_comparison['intersection'],
            intersection_file,
            pathseq_taxids=sample_comparison['pathseq_taxids_for_intersection'],
            compress=True
)
# Save sample-level MiTRI-only reads
if sample_comparison['mitri_only']:
    mitri_only_file = sample_level_dir / "mitri_only_reads.txt"
    save_reads_to_file(
        sample_comparison['mitri_only'],
        mitri_only_file,
        compress=True
)
logger.info(f" Sample-level: Total_MiTRI={len(all_mitri_reads)}, "
    f"Intersection={sample_comparison['intersection_count']} ({sample_comparison['intersection_pct']:.2f}%), "
    f"MiTRI_only={sample_comparison['mitri_only_count']} ({sample_comparison['mitri_only_pct']:.2f}%)")
return sample_results


# ==================== textpysamtext ====================

def test_pysam_consistency(cancer: str, status: str, sample: str, logger: logging.Logger):
    """
    Test consistency between pysam and samtools extraction results
    """
    logger.info(f"\n{'='*60}")
    logger.info(f"Testing pysam consistency for {cancer}/{status}/{sample}")
    logger.info(f"{'='*60}")
    # PathSeq BAMpath
    pathseq_bam = Path(PATHSEQ_BASE) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.error(f"BAM file not found: {pathseq_bam}")
        return False
    # textsamtoolstext
    txt_path = pathseq_bam.parent / f"{sample}_temp_test.txt"
    cmd = f"conda run -n deeptools_env samtools view -h {pathseq_bam} > {txt_path}"
    logger.info(f"Running: {cmd}")
    os.system(cmd)
    if not txt_path.exists():
        logger.error("Failed to create txt file")
        return False
    # extractreads
    pysam_reads = extract_pathseq_reads_pysam(pathseq_bam)
    samtools_reads = extract_pathseq_reads_samtools(txt_path)
    # compare
    pysam_keys = set(pysam_reads.keys())
    samtools_keys = set(samtools_reads.keys())
    only_pysam = pysam_keys - samtools_keys
    only_samtools = samtools_keys - pysam_keys
    common = pysam_keys & samtools_keys
    logger.info(f"\nComparison results:")
    logger.info(f" pysam reads: {len(pysam_reads)}")
    logger.info(f" samtools reads: {len(samtools_reads)}")
    logger.info(f" Common reads: {len(common)}")
    logger.info(f" Only in pysam: {len(only_pysam)}")
    logger.info(f" Only in samtools: {len(only_samtools)}")
    # checktaxidtext
    taxid_mismatch = 0
    for key in list(common)[:100]: # checkfirst100
        if pysam_reads[key] != samtools_reads[key]:
            taxid_mismatch += 1
            logger.info(f" Taxid mismatches in first 100 common reads: {taxid_mismatch}")
            # textfile
            txt_path.unlink()
            # text
            consistency = len(only_pysam) == 0 and len(only_samtools) == 0 and taxid_mismatch == 0
            if consistency:
                logger.info("\nOK pysam extraction is CONSISTENT with samtools!")
            else:
                logger.warning("\nFailed pysam extraction has DIFFERENCES with samtools!")
                return consistency


# ==================== resultssave ====================

def save_results(all_results: List[Dict], cancer: str, output_dir: Path):
    """Save statistics results (detailed reads were already saved in process_sample)"""
    stats_dir = output_dir / STATISTICS_DIR / cancer
    stats_dir.mkdir(parents=True, exist_ok=True)
    # Task 1: microbe levelresults
    microbe_records = []
    for result in all_results:
        if not result:
            continue
        base_info = {
            'cancer': result['cancer'],
            'status': result['status'],
            'sample': result['sample'],
            'pathseq_reads_count': result['pathseq_reads_count']
        }
        for mr in result['microbe_results']:
            record = {
                **base_info,
                'microbe': mr['microbe'],
                'species_taxid': mr['species_taxid'],
                'tax_ids': ','.join(map(str, mr['tax_ids'])),
                'genus_taxid': mr['genus_taxid'],
                'mitri_reads_count': mr['mitri_reads_count'],
                'intersection_count': mr['intersection_count'],
                'intersection_pct': round(mr['intersection_pct'], 2),
                'mitri_only_count': mr['mitri_only_count'],
                'mitri_only_pct': round(mr['mitri_only_pct'], 2),
                'taxid_matched_count': mr.get('taxid_matched_count', 0),
                'taxid_matched_pct': round(mr.get('taxid_matched_pct', 0), 2)
            }
            microbe_records.append(record)
            if microbe_records:
                df_microbe = pd.DataFrame(microbe_records)
                df_microbe.to_csv(stats_dir / f"{cancer}_microbe_level.csv", index=False)
                # Task 2: sample levelresults
                sample_records = []
                for result in all_results:
                    if not result or not result['sample_level_result']:
                        continue
                    slr = result['sample_level_result']
                    record = {
                        'cancer': result['cancer'],
                        'status': result['status'],
                        'sample': result['sample'],
                        'pathseq_reads_count': result['pathseq_reads_count'],
                        'total_mitri_reads': slr['total_mitri_reads'],
                        'intersection_count': slr['intersection_count'],
                        'intersection_pct': round(slr['intersection_pct'], 2),
                        'mitri_only_count': slr['mitri_only_count'],
                        'mitri_only_pct': round(slr['mitri_only_pct'], 2)
                    }
                    sample_records.append(record)
                    if sample_records:
                        df_sample = pd.DataFrame(sample_records)
                        df_sample.to_csv(stats_dir / f"{cancer}_sample_level.csv", index=False)


# ==================== text ====================

def process_cancer(cancer: str, microbe_taxid_map: Dict, test_consistency: bool = False):
    """processtextcancer type"""
    # textlogger
    log_dir = Path(OUTPUT_BASE) / LOGS_DIR
    logger = setup_logger(cancer, log_dir)
    logger.info(f"\n{'='*80}")
    logger.info(f"Processing cancer type: {cancer}")
    logger.info(f"{'='*80}\n")
    # loadmicrobe list
    microbes = load_microbe_list(cancer)
    if not microbes:
        logger.warning(f"No microbes found for {cancer}")
        return
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textpysamtext(optional)
    if test_consistency:
        # Select the first available sample for testing
        for status in STATUSES:
            bam_dir = Path(PATHSEQ_BASE) / cancer / status / "bam"
            if bam_dir.exists():
                bam_files = list(bam_dir.glob("*_rna_output.pathseq.bam"))
                if bam_files:
                    test_sample = bam_files[0].stem.replace("_rna_output.pathseq", "")
                    test_pysam_consistency(cancer, status, test_sample, logger)
                    break
                # Collect all samples
                all_samples = []
                for status in STATUSES:
                    bam_dir = Path(PATHSEQ_BASE) / cancer / status / "bam"
                    if not bam_dir.exists():
                        continue
                    bam_files = bam_dir.glob("*_rna_output.pathseq.bam")
                    for bam_file in bam_files:
                        sample = bam_file.stem.replace("_rna_output.pathseq", "")
                        all_samples.append((cancer, status, sample, microbes, microbe_taxid_map))
                        logger.info(f"Found {len(all_samples)} samples to process")
                        # textrowsProcess sample
                        all_results = []
                        with ProcessPoolExecutor(max_workers=MAX_WORKERS) as executor:
                            futures = {executor.submit(process_sample, args): args[2] for args in all_samples}
                            for future in as_completed(futures):
                                sample = futures[future]
                                try:
                                    result = future.result()
                                    if result:
                                        all_results.append(result)
                                except Exception as e:
                                    logger.error(f"Error processing sample {sample}: {e}")
                                    # savestatisticsresults
                                    output_dir = Path(OUTPUT_BASE)
                                    save_results(all_results, cancer, output_dir)
                                    logger.info(f"\nCompleted {cancer}: processed {len(all_results)} samples")


def main():
    """text"""
    print("="*80)
    print("PathSeq vs MiTRI Reads Comparison Pipeline")
    print("="*80)
    # Create output directory
    Path(OUTPUT_BASE).mkdir(parents=True, exist_ok=True)
    (Path(OUTPUT_BASE) / STATISTICS_DIR).mkdir(parents=True, exist_ok=True)
    (Path(OUTPUT_BASE) / DETAILED_READS_DIR).mkdir(parents=True, exist_ok=True)
    (Path(OUTPUT_BASE) / LOGS_DIR).mkdir(parents=True, exist_ok=True)
    # loadtaxidtextinformation
    print("\nLoading taxid hierarchy...")
    taxid_df = load_taxid_hierarchy()
    microbe_taxid_map = build_microbe_taxid_map(taxid_df)
    print(f"Loaded {len(microbe_taxid_map)} microbe taxid mappings")
    # processeachcancer type
    print(f"\nProcessing {len(CANCERS)} cancer types...")
    print(f"Using {MAX_WORKERS} parallel workers")
    for cancer in CANCERS:
        try:
            # Test pysam consistency only for the first cancer type
            test_consistency = (cancer == CANCERS[0])
            process_cancer(cancer, microbe_taxid_map, test_consistency)
        except Exception as e:
            print(f"Error processing {cancer}: {e}")
            continue
        # Summarize all results
        print("\nGenerating summary reports...")
        # Summarize microbe-level results
        all_microbe_dfs = []
        for cancer in CANCERS:
            csv_path = Path(OUTPUT_BASE) / STATISTICS_DIR / cancer / f"{cancer}_microbe_level.csv"
            if csv_path.exists():
                df = pd.read_csv(csv_path)
                all_microbe_dfs.append(df)
                if all_microbe_dfs:
                    summary_microbe = pd.concat(all_microbe_dfs, ignore_index=True)
                    summary_microbe.to_csv(Path(OUTPUT_BASE) / STATISTICS_DIR / "summary_all_microbe_level.csv", index=False)
                    print(f" Microbe-level summary: {len(summary_microbe)} records")
                    # Summarize sample-level results
                    all_sample_dfs = []
                    for cancer in CANCERS:
                        csv_path = Path(OUTPUT_BASE) / STATISTICS_DIR / cancer / f"{cancer}_sample_level.csv"
                        if csv_path.exists():
                            df = pd.read_csv(csv_path)
                            all_sample_dfs.append(df)
                            if all_sample_dfs:
                                summary_sample = pd.concat(all_sample_dfs, ignore_index=True)
                                summary_sample.to_csv(Path(OUTPUT_BASE) / STATISTICS_DIR / "summary_all_sample_level.csv", index=False)
                                print(f" Sample-level summary: {len(summary_sample)} records")
                                print("\n" + "="*80)
                                print("Pipeline completed successfully!")
                                print(f"Results saved to: {OUTPUT_BASE}")
                                print(f" - Statistics: {OUTPUT_BASE}/{STATISTICS_DIR}")
                                print(f" - Detailed reads: {OUTPUT_BASE}/{DETAILED_READS_DIR}")
                                print(f" - Logs: {OUTPUT_BASE}/{LOGS_DIR}")
                                print("="*80)


if __name__ == "__main__":
    main()

# code V2

In [ ]:
#!/usr/bin/env python3
"""
BAM read intersection statistics tool
Compare read intersections between PathSeq and MiTRI
"""

import os
import sys
import pysam
import logging
import pandas as pd
from pathlib import Path
from collections import defaultdict
from multiprocessing import Pool, Manager
from datetime import datetime
import traceback

# Global configuration
# CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
CANCERS = ["THCA"]
BASE_PATHSEQ_DIR = "/path/to/project/data/cancers_V1"
BASE_MITRI_DIR = "/path/to/project/results/cancers_V1_V2.8"
CSV_DIR = "/path/to/project/data/CSVs_20251203"
TAXID_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
OUTPUT_DIR = "/path/to/project/results/summary_V2/reads_compare"
N_PROCESSES = 64 # reserve 10 cores for the system


class TaxidMapper:
    """Taxidtextmappingtext"""
    def __init__(self, taxid_file):
        self.df = pd.read_csv(taxid_file, sep='\t')
        self.microbe_to_taxids = {}
        self._build_mapping()
        def _normalize_name(self, name):
            """textmicrobetext"""
            return name.replace(' ', '_').replace('/', '_').replace('\\', '_')
        def _build_mapping(self):
            """buildmicrobetexttotaxidmapping"""
            for _, row in self.df.iterrows():
                if pd.notna(row['species_scientific_name']):
                    normalized_name = self._normalize_name(row['species_scientific_name'])
                    if normalized_name not in self.microbe_to_taxids:
                        self.microbe_to_taxids[normalized_name] = {
                            'species': set(),
                            'tax_ids': set(),
                            'genus': set()
                        }
                        # recordspeciestexttaxid
                        if pd.notna(row['species']):
                            self.microbe_to_taxids[normalized_name]['species'].add(int(row['species']))
                            # recordtexttax_id
                            if pd.notna(row['tax_id']):
                                self.microbe_to_taxids[normalized_name]['tax_ids'].add(int(row['tax_id']))
                                # recordgenustexttaxid
                                if pd.notna(row['genus']):
                                    self.microbe_to_taxids[normalized_name]['genus'].add(int(row['genus']))
                                    def get_taxids(self, microbe_name):
                                        """textmicrobetaxidinformation"""
                                        return self.microbe_to_taxids.get(microbe_name, {
                                        'species': set(),
                                        'tax_ids': set(),
                                        'genus': set()
                                    })


class ReadsExtractor:
    """Readsextracttext"""
    @staticmethod
    def extract_pathseq_reads(bam_path, logger):
        """
        Extract reads with YP tags from a PathSeq BAM file
        return: {(read_id, sequence): set(taxids)}
        """
        reads_dict = {}
        try:
            with pysam.AlignmentFile(bam_path, "rb") as bam:
                for read in bam:
                    if read.has_tag('YP'):
                        yp_value = read.get_tag('YP')
                        taxids = set(int(x) for x in str(yp_value).split(',') if x.strip())
                        # textread_idandsequencetextforkey
                        key = (read.query_name, read.query_sequence)
                        if key in reads_dict:
                            reads_dict[key].update(taxids)
                        else:
                            reads_dict[key] = taxids
                            logger.info(f"PathSeq BAMextractcompleted: {len(reads_dict)} unique reads")
                            return reads_dict
        except Exception as e:
            logger.error(f"extractPathSeq readsfailed {bam_path}: {e}")
            return {}
        @staticmethod
        def extract_mitri_reads(txt_path, logger):
            """
            Extract reads from MiTRI txt files
            return: set((read_id, sequence))
            """
            reads_set = set()
            try:
                with open(txt_path, 'r') as f:
                    for line in f:
                        if line.startswith('@'): # skipheader
                            continue
                        fields = line.strip().split('\t')
                        if len(fields) < 10:
                            continue
                        read_id = fields[0]
                        sequence = fields[9]
                        reads_set.add((read_id, sequence))
                        logger.info(f"MiTRI txtextractcompleted: {len(reads_set)} reads")
                        return reads_set
            except Exception as e:
                logger.error(f"Extract MiTRI readsfailed {txt_path}: {e}")
                return set()


class ReadsComparator:
    """Readscomparetext"""
    @staticmethod
    def compare_reads(mitri_reads, pathseq_reads):
        """
        Compare MiTRI and PathSeq reads
        return: (intersection, mitri_only, stats)
        """
        mitri_keys = set(mitri_reads) if isinstance(mitri_reads, set) else set(mitri_reads.keys())
        pathseq_keys = set(pathseq_reads.keys())
        intersection = mitri_keys & pathseq_keys
        mitri_only = mitri_keys - pathseq_keys
        total_mitri = len(mitri_keys)
        n_intersection = len(intersection)
        n_mitri_only = len(mitri_only)
        stats = {
            'total_mitri_reads': total_mitri,
            'intersection_reads': n_intersection,
            'intersection_pct': (n_intersection / total_mitri * 100) if total_mitri > 0 else 0,
            'mitri_only_reads': n_mitri_only,
            'mitri_only_pct': (n_mitri_only / total_mitri * 100) if total_mitri > 0 else 0
        }
        return intersection, mitri_only, stats
    @staticmethod
    def check_taxid_match(reads_keys, pathseq_reads, target_taxids):
        """
        Check whether read taxids match
        return: set of matched reads keys
        """
        matched = set()
        for key in reads_keys:
            if key in pathseq_reads:
                pathseq_taxids = pathseq_reads[key]
                if pathseq_taxids & target_taxids: # withtextformatched
                    matched.add(key)
                    return matched


def process_sample(args):
    """
    Process all tasks for one sample
    """
    cancer, status, sample, microbe_list, taxid_mapper, log_queue = args
    # textlogger
    logger = logging.getLogger(f"{cancer}_{sample}")
    logger.setLevel(logging.INFO)
    # fileprocesstext
    log_dir = Path(OUTPUT_DIR) / cancer / "logs"
    log_dir.mkdir(parents=True, exist_ok=True)
    fh = logging.FileHandler(log_dir / f"{sample}.log")
    fh.setFormatter(logging.Formatter('%(asctime)s - %(levelname)s - %(message)s'))
    logger.addHandler(fh)
    logger.info(f"startProcess sample: {cancer}/{status}/{sample}")
    try:
        # 1. extractPathSeq reads
        pathseq_bam = Path(BASE_PATHSEQ_DIR) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
        if not pathseq_bam.exists():
            logger.warning(f"PathSeq BAMtextexists: {pathseq_bam}")
            return None
        pathseq_reads = ReadsExtractor.extract_pathseq_reads(str(pathseq_bam), logger)
        if not pathseq_reads:
            logger.warning(f"PathSeq readsfortext")
            return None
        # 2. processtextmicrobes (text1andTask 3)
        task1_results = []
        task3_results = []
        all_mitri_reads = set() # text2
        for microbe in microbe_list:
            logger.info(f"processmicrobe: {microbe}")
            # Extract MiTRI reads
            mitri_txt = Path(BASE_MITRI_DIR) / cancer / "03.coverage" / status / microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt"
            if not mitri_txt.exists():
                logger.warning(f"MiTRI txttextexists: {mitri_txt}")
                continue
            mitri_reads = ReadsExtractor.extract_mitri_reads(str(mitri_txt), logger)
            if not mitri_reads:
                logger.warning(f"MiTRI readsfortext: {microbe}")
                continue
            # text1: textcompare
            intersection, mitri_only, stats = ReadsComparator.compare_reads(mitri_reads, pathseq_reads)
            # Get microbe taxid information
            taxid_info = taxid_mapper.get_taxids(microbe)
            task1_result = {
                'cancer': cancer,
                'status': status,
                'sample': sample,
                'microbe': microbe,
                'species_taxids': ','.join(map(str, sorted(taxid_info['species']))),
                'tax_ids': ','.join(map(str, sorted(taxid_info['tax_ids']))),
                'genus_taxids': ','.join(map(str, sorted(taxid_info['genus']))),
                **stats
            }
            task1_results.append(task1_result)
            # savedetailedreadsinformation
            save_detailed_reads(cancer, sample, microbe, "task1",                 intersection, mitri_only, logger)
            # Task 3: texttaxidmatched
            target_taxids = taxid_info['tax_ids']
            if target_taxids:
                matched_intersection = ReadsComparator.check_taxid_match(
                    intersection, pathseq_reads, target_taxids
)
matched_mitri_only = ReadsComparator.check_taxid_match(
    mitri_only, pathseq_reads, target_taxids
)
task3_result = {
    **task1_result,
    'taxid_matched_intersection': len(matched_intersection),
    'taxid_matched_intersection_pct': (len(matched_intersection) / len(intersection) * 100) if intersection else 0,
    'taxid_matched_mitri_only': len(matched_mitri_only),
    'taxid_matched_mitri_only_pct': (len(matched_mitri_only) / len(mitri_only) * 100) if mitri_only else 0
}
task3_results.append(task3_result)
# savedetailedreadsinformation
save_detailed_reads(cancer, sample, microbe, "task3",     matched_intersection, matched_mitri_only, logger)
# textallMiTRI readstext2
all_mitri_reads.update(mitri_reads)
# 3. text2: sampletextcompare
if all_mitri_reads:
    logger.info(f"text2: sampletext, textreadstext: {len(all_mitri_reads)}")
    intersection, mitri_only, stats = ReadsComparator.compare_reads(
        all_mitri_reads, pathseq_reads
)
task2_result = {
    'cancer': cancer,
    'status': status,
    'sample': sample,
    'n_microbes': len(microbe_list),
    **stats
}
# savedetailedreadsinformation
save_detailed_reads(cancer, sample, "aggregated", "task2",     intersection, mitri_only, logger)
else:
    task2_result = None
    logger.info(f"sampleProcessing completed: {sample}")
    return {
        'task1': task1_results,
        'task2': task2_result,
        'task3': task3_results
    }
    except Exception as e:
        logger.error(f"Process samplefailed: {sample}")
        logger.error(traceback.format_exc())
        return None


def save_detailed_reads(cancer, sample, microbe, task, intersection, only_set, logger):
    """savedetailedreadsinformation"""
    try:
        output_dir = Path(OUTPUT_DIR) / cancer / "details" / task
        output_dir.mkdir(parents=True, exist_ok=True)
        # savetextreads
        if intersection:
            inter_file = output_dir / f"{sample}_{microbe}_intersection.txt"
            with open(inter_file, 'w') as f:
                f.write("read_id\tsequence\n")
                for read_id, seq in sorted(intersection):
                    f.write(f"{read_id}\t{seq}\n")
                    # savetextwithreads
                    if only_set:
                        only_file = output_dir / f"{sample}_{microbe}_mitri_only.txt"
                        with open(only_file, 'w') as f:
                            f.write("read_id\tsequence\n")
                            for read_id, seq in sorted(only_set):
                                f.write(f"{read_id}\t{seq}\n")
    except Exception as e:
        logger.error(f"savedetailedreadsfailed: {e}")


def get_sample_list(cancer, status):
    """textsample columntext"""
    bam_dir = Path(BASE_PATHSEQ_DIR) / cancer / status / "bam"
    if not bam_dir.exists():
        return []
    samples = []
    for bam_file in bam_dir.glob("*_rna_output.pathseq.bam"):
        sample_name = bam_file.name.replace("_rna_output.pathseq.bam", "")
        samples.append(sample_name)
        return samples


def get_microbe_list(cancer):
    """textmicrobe list"""
    csv_file = Path(CSV_DIR) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    if 'Taxon' not in df.columns:
        return []
    return df['Taxon'].tolist()


def main():
    """text"""
    print("=" * 80)
    print("BAM read intersection statistics tool")
    print("=" * 80)
    print(f"starttext: {datetime.now()}")
    print(f"textrowstext: {N_PROCESSES}")
    print()
    # Create output directory
    Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
    # loadtaxidmapping
    print("loadTaxidtextmapping...")
    taxid_mapper = TaxidMapper(TAXID_FILE)
    print(f"textload {len(taxid_mapper.microbe_to_taxids)} microbestaxidmapping")
    print()
    # textalltext
    all_tasks = []
    for cancer in CANCERS:
        print(f"text {cancer} text...")
        microbe_list = get_microbe_list(cancer)
        if not microbe_list:
            print(f" warning: not found {cancer} microbe list")
            continue
        print(f" microbecount: {len(microbe_list)}")
        for status in ['Normal', 'Tumor']:
            samples = get_sample_list(cancer, status)
            print(f" {status} sample count: {len(samples)}")
            for sample in samples:
                all_tasks.append((cancer, status, sample, microbe_list, taxid_mapper, None))
                print(f"\ntext: {len(all_tasks)}")
                print("=" * 80)
                print()
                # textrowsprocess
                print(f"starttextrowsprocess (text: {N_PROCESSES})...")
                all_results = {
                    'task1': [],
                    'task2': [],
                    'task3': []
                }
                with Pool(processes=N_PROCESSES) as pool:
                    for i, result in enumerate(pool.imap_unordered(process_sample, all_tasks), 1):
                        if result:
                            all_results['task1'].extend(result['task1'])
                            if result['task2']:
                                all_results['task2'].append(result['task2'])
                                all_results['task3'].extend(result['task3'])
                                if i % 10 == 0:
                                    print(f"text: {i}/{len(all_tasks)} ({i/len(all_tasks)*100:.1f}%)")
                                    print("\nProcessing completed, textgeneratetext...")
                                    # savetextresults
                                    for task_name, results in all_results.items():
                                        if results:
                                            df = pd.DataFrame(results)
                                            output_file = Path(OUTPUT_DIR) / f"summary_{task_name}.tsv"
                                            df.to_csv(output_file, sep='\t', index=False)
                                            print(f" {task_name} text: {len(results)} records -> {output_file}")
                                            print()
                                            print("=" * 80)
                                            print(f"completedtext: {datetime.now()}")
                                            print(f"resultssavetext: {OUTPUT_DIR}")
                                            print("=" * 80)


if __name__ == "__main__":
    main()

# Separate tasks

## task 1

### test

In [ ]:
#!/usr/bin/env python3
"""
Single-sample test script for validating logic
Quick test for processing one sample
"""

import os
from pathlib import Path
import pysam
import pandas as pd
from collections import defaultdict

# text
TEST_CANCER = "BRCA"
TEST_STATUS = "Normal"
TEST_SAMPLE = "SRR11600329" # textforactualexistssample

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
TAXID_HIERARCHY = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def test_taxid_mapping():
    """texttaxidmappingtext"""
    print("="*60)
    print("text1: Taxidmapping")
    print("="*60)
    df = pd.read_csv(TAXID_HIERARCHY, sep='\t')
    print(f"OK successfully readV7_taxid_hierarchy.txt: {len(df)} records")
    print(f"OK column names: {df.columns.tolist()}")
    # textmicrobetext
    test_names = df['species_scientific_name'].dropna().head(5)
    print(f"\ntextscientific names:")
    for name in test_names:
        converted = name.replace(' ', '_').replace('/', '_')
        print(f" {name} -> {converted}")
        # buildmapping
        mapping = defaultdict(list)
        for _, row in df.iterrows():
            if pd.notna(row['species_scientific_name']):
                microbe_name = row['species_scientific_name'].replace(' ', '_').replace('/', '_')
                mapping[microbe_name].append({
                    'tax_id': row['tax_id'],
                    'species': row['species'],
                    'genus': row['genus']
                })
                print(f"\nOK createmapping: {len(mapping)} microbes")
                # textmicrobes
                if mapping:
                    test_microbe = list(mapping.keys())[0]
                    print(f"\ntextmapping - {test_microbe}:")
                    for info in mapping[test_microbe]:
                        print(f" tax_id={info['tax_id']}, species={info['species']}, genus={info['genus']}")
                        return True


def test_pathseq_bam():
    """textPathSeq BAMread"""
    print("\n" + "="*60)
    print("text2: PathSeq BAMread")
    print("="*60)
    bam_path = Path(BASE_PATHSEQ) / TEST_CANCER / TEST_STATUS / "bam" / f"{TEST_SAMPLE}_rna_output.pathseq.bam"
    print(f"BAMpath: {bam_path}")
    if not bam_path.exists():
        print(f"Failed file does not exist, trying to list directory contents:")
        bam_dir = bam_path.parent
        if bam_dir.exists():
            files = list(bam_dir.glob("*.bam"))
            print(f" found {len(files)} BAMfile")
            if files:
                print(f" text: {files[0].name}")
                return False
            print(f"OK file exists")
            try:
                with pysam.AlignmentFile(str(bam_path), "rb") as bam:
                    total_reads = 0
                    yp_reads = 0
                    taxid_examples = []
                    max_reads_to_test = 10000
                    for read in bam:
                        # textchecktexttotext
                        if total_reads >= max_reads_to_test:
                            break
                        total_reads += 1
                        if read.has_tag('YP'):
                            yp_reads += 1
                            if len(taxid_examples) < 25:
                                yp_value = read.get_tag('YP')
                                taxid_examples.append({
                                    'read_id': read.query_name,
                                    'yp_tag': yp_value,
                                    'sequence_len': len(read.query_sequence) if read.query_sequence else 0
                                })
                                print(f"OK successfully read BAM file")
                                print(f" textfirst{total_reads}text: textreads={total_reads}, textYPtext={yp_reads}")
                                print(f" YPtextcoveragetext: {yp_reads/total_reads*100:.1f}%")
                                print(f" YPtext (first{len(taxid_examples)}):")
                                for ex in taxid_examples:
                                    print(f" {ex['read_id']}: YP={ex['yp_tag']}, seq_len={ex['sequence_len']}")
                                    return True
            except Exception as e:
                print(f"Failed readBAMfailed: {e}")
                return False


def test_mitri_txt():
    """textMiTRI txtread"""
    print("\n" + "="*60)
    print("text3: MiTRI txtread")
    print("="*60)
    # readmicrobe list
    csv_path = Path(BASE_CSV) / f"{TEST_CANCER}_V3.csv"
    if not csv_path.exists():
        print(f"Failed CSV file does not exist: {csv_path}")
        return False
    df_microbes = pd.read_csv(csv_path)
    microbes = df_microbes['Taxon'].tolist()
    print(f"OK readmicrobe list: {len(microbes)} ")
    print(f" text: {microbes[:3]}")
    # textmicrobes
    test_microbe = microbes[0]
    txt_path = (Path(BASE_MITRI) / TEST_CANCER / "03.coverage" / TEST_STATUS /         test_microbe / f"{TEST_SAMPLE}_rna_output.pathseq_sorted.bam.txt")
    print(f"\ntextmicrobe: {test_microbe}")
    print(f"txtpath: {txt_path}")
    if not txt_path.exists():
        print(f"Failed filetextexists")
        # textfindtextfile
        microbe_dir = txt_path.parent
        if microbe_dir.exists():
            files = list(microbe_dir.glob("*.txt"))
            print(f" textmicrobedirectorytextwith {len(files)} txtfile")
            return False
        print(f"OK file exists")
        try:
            with open(txt_path, 'r') as f:
                header_lines = 0
                read_lines = 0
                examples = []
                max_reads_to_test = 100
                for line in f:
                    if line.startswith('@'):
                        header_lines += 1
                    else:
                        # textchecktexttotext
                        if read_lines >= max_reads_to_test:
                            break
                        read_lines += 1
                        if len(examples) < 3:
                            fields = line.strip().split('\t')
                            if len(fields) >= 10:
                                examples.append({
                                    'read_id': fields[0],
                                    'flag': fields[1],
                                    'sequence': fields[9][:30] + '...'
                                })
                                print(f"OK successfully read txt file")
                                print(f" headerrowstext={header_lines}, readrowstext={read_lines}")
                                print(f" readstext:")
                                for ex in examples:
                                    print(f" {ex['read_id']}: flag={ex['flag']}, seq={ex['sequence']}")
                                    return True
        except Exception as e:
            print(f"Failed readtxtfailed: {e}")
            return False


def test_comparison():
    """textreadscomparetext"""
    print("\n" + "="*60)
    print("text4: Readscomparetext")
    print("="*60)
    # createtextdata
    pathseq_reads = {
        ('read1', 'ATCG'),
        ('read2', 'GCTA'),
        ('read3', 'TTAA'),
        ('read4', 'CCGG')
    }
    mitri_reads = {
        ('read1', 'ATCG'), # text
        ('read2', 'GCTA'), # text
        ('read5', 'AAAA'), # MiTRItextwith
        ('read6', 'GGGG') # MiTRItextwith
    }
    intersection = mitri_reads & pathseq_reads
    mitri_only = mitri_reads - pathseq_reads
    print(f"PathSeq reads: {len(pathseq_reads)}")
    print(f"MiTRI reads: {len(mitri_reads)}")
    print(f"text: {len(intersection)} ({len(intersection)/len(mitri_reads)*100:.1f}%)")
    print(f" {intersection}")
    print(f"MiTRItextwith: {len(mitri_only)} ({len(mitri_only)/len(mitri_reads)*100:.1f}%)")
    print(f" {mitri_only}")
    assert len(intersection) == 2
    assert len(mitri_only) == 2
    print("OK comparetextcorrect")
    return True


def main():
    """textrowsalltext"""
    print("\n" + "="*70)
    print("textsampletext")
    print("textsample: {}/{}/{}".format(TEST_CANCER, TEST_STATUS, TEST_SAMPLE))
    print("="*70 + "\n")
    results = []
    # textrowstext
    results.append(("Taxidmapping", test_taxid_mapping()))
    results.append(("PathSeq BAMread", test_pathseq_bam()))
    results.append(("MiTRI txtread", test_mitri_txt()))
    results.append(("Readscomparetext", test_comparison()))
    # text
    print("\n" + "="*70)
    print("text")
    print("="*70)
    for name, result in results:
        status = "OK passed" if result else "Failed failed"
        print(f"{name}: {status}")
        all_passed = all(r[1] for r in results)
        if all_passed:
            print("\n🎉 alltextpassed! textrowstext.")
        else:
            print("\nWarning️ textfailed, textcheckpathandfiletextexists.")
            print("="*70 + "\n")


if __name__ == "__main__":
    main()

### Main script

In [ ]:
#!/usr/bin/env python3
"""
PathSeq and MiTRI read intersection analysis script
textrowsprocesstextcancer typetextsample counttext
"""

import os
import re
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, Tuple, List
import pandas as pd
import pysam
from multiprocessing import Pool, Manager
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
# CANCERS = ["THCA"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 64 # reserve 5 of 80 cores for the system

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
TAXID_HIERARCHY = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
OUTPUT_BASE = "/path/to/project/results/summary_V2/reads_compare"


class TaxidMapper:
    """Taxidmappingtext"""
    def __init__(self, hierarchy_file: str):
        self.df = pd.read_csv(hierarchy_file, sep='\t')
        self.microbe_to_taxids = self._build_microbe_mapping()
        def _build_microbe_mapping(self) -> Dict[str, List[Dict]]:
            """buildmicrobetexttotaxidmapping"""
            mapping = defaultdict(list)
            for _, row in self.df.iterrows():
                if pd.notna(row['species_scientific_name']):
                    # textscientific nametextforfiletext
                    microbe_name = re.sub(r'[/\s]+', '_', str(row['species_scientific_name']))
                    mapping[microbe_name].append({
                        'tax_id': int(row['tax_id']),
                        'species': int(row['species']) if pd.notna(row['species']) else None,
                        'genus': int(row['genus']) if pd.notna(row['genus']) else None
                    })
                    return dict(mapping)
                def get_taxids(self, microbe_name: str) -> List[Dict]:
                    """textmicrobetextalltaxidinformation"""
                    return self.microbe_to_taxids.get(microbe_name, [])


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(cancer)
    logger.setLevel(logging.INFO)
    # clear existing handlers
    if logger.handlers:
        logger.handlers.clear()
        # file handler
        fh = logging.FileHandler(output_dir / f"{cancer}_processing.log")
        fh.setLevel(logging.INFO)
        # format
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_pathseq_reads(bam_path: str) -> Tuple[Set[Tuple[str, str]], Dict[Tuple[str, str], Set[int]]]:
    """
    Extract reads with YP tags from a PathSeq BAM file
    Returns:
    reads_set: {(read_id, sequence), ...}
    reads_taxids: {(read_id, sequence): {taxid1, taxid2, ...}, ...}
    """
    reads_set = set()
    reads_taxids = defaultdict(set)
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bam:
            for read in bam:
                if read.has_tag('YP'):
                    read_id = read.query_name
                    sequence = read.query_sequence
                    if sequence is None:
                        continue
                    key = (read_id, sequence)
                    reads_set.add(key)
                    # extracttaxids
                    yp_value = read.get_tag('YP')
                    taxids = [int(t.strip()) for t in str(yp_value).split(',')]
                    reads_taxids[key].update(taxids)
    except Exception as e:
        logging.error(f"Error reading PathSeq BAM {bam_path}: {e}")
        return reads_set, dict(reads_taxids)


def extract_mitri_reads(txt_path: str) -> Set[Tuple[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files
    Returns:
    reads_set: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                if sequence != '*':
                    reads_set.add((read_id, sequence))
    except FileNotFoundError:
        logging.warning(f"MiTRI file not found: {txt_path}")
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_set


def compare_reads(mitri_reads: Set[Tuple[str, str]],     pathseq_reads: Set[Tuple[str, str]]) -> Dict:
"""
Compare MiTRI and PathSeq reads
Returns:
{
'intersection': set,
'mitri_only': set,
'intersection_count': int,
'intersection_pct': float,
'mitri_only_count': int,
'mitri_only_pct': float
}
"""
intersection = mitri_reads & pathseq_reads
mitri_only = mitri_reads - pathseq_reads
total_mitri = len(mitri_reads)
return {
    'intersection': intersection,
    'mitri_only': mitri_only,
    'intersection_count': len(intersection),
    'intersection_pct': (len(intersection) / total_mitri * 100) if total_mitri > 0 else 0,
    'mitri_only_count': len(mitri_only),
    'mitri_only_pct': (len(mitri_only) / total_mitri * 100) if total_mitri > 0 else 0,
    'total_mitri': total_mitri,
    'total_pathseq': len(pathseq_reads)
}


def process_sample(args):
    """processtextsamplesallmicrobe"""
    cancer, status, sample, microbes, taxid_mapper, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # PathSeq BAMpath
    pathseq_bam = Path(BASE_PATHSEQ) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.warning(f"PathSeq BAM not found: {pathseq_bam}")
        return []
    # extractPathSeq reads
    pathseq_reads, pathseq_taxids = extract_pathseq_reads(str(pathseq_bam))
    logger.info(f"{sample}: PathSeq reads = {len(pathseq_reads)}")
    results = []
    # processtextmicrobes
    for microbe in microbes:
        mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /             microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
        if not mitri_txt.exists():
            continue
        # Extract MiTRI reads
        mitri_reads = extract_mitri_reads(str(mitri_txt))
        if len(mitri_reads) == 0:
            continue
        # comparereads
        comparison = compare_reads(mitri_reads, pathseq_reads)
        # texttaxidinformation
        taxid_info = taxid_mapper.get_taxids(microbe)
        # savedetailedresults
        detail_file = output_dir / cancer / f"{sample}_{microbe}_task1_detail.tsv"
        detail_file.parent.mkdir(parents=True, exist_ok=True)
        with open(detail_file, 'w') as f:
            f.write("read_id\tsequence\ttype\n")
            for read_id, seq in comparison['intersection']:
                f.write(f"{read_id}\t{seq}\tintersection\n")
                for read_id, seq in comparison['mitri_only']:
                    f.write(f"{read_id}\t{seq}\tmitri_only\n")
                    # textinformation
                    for taxid_dict in taxid_info:
                        results.append({
                            'cancer': cancer,
                            'status': status,
                            'sample': sample,
                            'microbe': microbe,
                            'species_taxid': taxid_dict['species'],
                            'tax_id': taxid_dict['tax_id'],
                            'genus_taxid': taxid_dict['genus'],
                            'total_mitri_reads': comparison['total_mitri'],
                            'total_pathseq_reads': comparison['total_pathseq'],
                            'intersection_count': comparison['intersection_count'],
                            'intersection_pct': comparison['intersection_pct'],
                            'mitri_only_count': comparison['mitri_only_count'],
                            'mitri_only_pct': comparison['mitri_only_pct']
                        })
                        logger.info(f"Completed {cancer}/{status}/{sample}: {len(results)} microbes")
                        return results


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext"""
    bam_dir = Path(BASE_PATHSEQ) / cancer / status / "bam"
    if not bam_dir.exists():
        return []
    samples = []
    for bam_file in bam_dir.glob("*_rna_output.pathseq.bam"):
        sample = bam_file.name.replace("_rna_output.pathseq.bam", "")
        samples.append(sample)
        return samples


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, taxid_mapper: TaxidMapper, output_base: Path):
    """processtextcancer typealldata"""
    cancer_output = output_base / cancer
    cancer_output.mkdir(parents=True, exist_ok=True)
    # textlogger
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    # textmicrobe list
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, taxid_mapper,                 output_base, cancer))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_results = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for results in pool.imap_unordered(process_sample, tasks):
                    all_results.extend(results)
                    # savecancer typetext
                    if all_results:
                        df_cancer = pd.DataFrame(all_results)
                        summary_file = cancer_output / f"{cancer}_task1_summary.tsv"
                        df_cancer.to_csv(summary_file, sep='\t', index=False)
                        logger.info(f"Saved cancer summary: {summary_file}")
                        logger.info(f"=== Completed {cancer}: {len(all_results)} records ===")
                        return all_results


def create_visualizations(df: pd.DataFrame, output_dir: Path):
    """createtext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        # 1. Summarize by cancer typetext
        ax1 = axes[0, 0]
        cancer_stats = df.groupby('cancer')['intersection_pct'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average Intersection %')
        ax1.set_title('Average Intersection Percentage by Cancer Type')
        # 2. MiTRItextwithreadstext
        ax2 = axes[0, 1]
        df['mitri_only_pct'].hist(bins=50, ax=ax2, color='coral', edgecolor='black')
        ax2.set_xlabel('MiTRI Only %')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI-Only Reads Percentage')
        # 3. Statuscompare
        ax3 = axes[1, 0]
        status_stats = df.groupby('status')[['intersection_pct', 'mitri_only_pct']].mean()
        status_stats.plot(kind='bar', ax=ax3)
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Percentage')
        ax3.set_title('Intersection vs MiTRI-Only by Status')
        ax3.legend(['Intersection %', 'MiTRI Only %'])
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        # 4. Top 20microbetext
        ax4 = axes[1, 1]
        top_microbes = df.groupby('microbe')['intersection_count'].sum().nlargest(20)
        top_microbes.plot(kind='barh', ax=ax4, color='green')
        ax4.set_xlabel('Total Intersection Count')
        ax4.set_title('Top 20 Microbes by Intersection Count')
        plt.tight_layout()
        plt.savefig(output_dir / 'task1_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("Visualization saved successfully")
    except ImportError:
        print("Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Error creating visualization: {e}")


def main():
    """text"""
    start_time = time.time()
    print(f"=== Reads Comparison Analysis Started at {datetime.now()} ===")
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # texttaxidmapping
    print("Loading taxid hierarchy...")
    taxid_mapper = TaxidMapper(TAXID_HIERARCHY)
    print(f"Loaded {len(taxid_mapper.microbe_to_taxids)} microbe mappings")
    # processeachcancer type
    all_results = []
    for cancer in CANCERS:
        print(f"\n{'='*60}")
        print(f"Processing {cancer}...")
        print(f"{'='*60}")
        cancer_results = process_cancer(cancer, taxid_mapper, output_base)
        all_results.extend(cancer_results)
        # savetext
        if all_results:
            df_all = pd.DataFrame(all_results)
            summary_file = output_base / "all_samples_task1_summary.tsv"
            df_all.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*60)
            print("OVERALL STATISTICS")
            print("="*60)
            print(f"Total records: {len(df_all)}")
            print(f"Total samples: {df_all['sample'].nunique()}")
            print(f"Total microbes: {df_all['microbe'].nunique()}")
            print(f"\nAverage intersection: {df_all['intersection_pct'].mean():.2f}%")
            print(f"Average MiTRI only: {df_all['mitri_only_pct'].mean():.2f}%")
            print(f"\nBy Cancer Type:")
            print(df_all.groupby('cancer')[['intersection_pct', 'mitri_only_pct']].mean().round(2))
            print(f"\nBy Status:")
            print(df_all.groupby('status')[['intersection_pct', 'mitri_only_pct']].mean().round(2))
            # createtext
            print("\nGenerating visualizations...")
            create_visualizations(df_all, output_base)
            elapsed_time = time.time() - start_time
            print(f"\n{'='*60}")
            print(f"=== Analysis Completed in {elapsed_time/60:.2f} minutes ===")
            print(f"{'='*60}")


if __name__ == "__main__":
    main()

## task 2

### test

In [ ]:
#!/usr/bin/env python3
"""
Task 2 single-sample test script, validate sample-level aggregation logic
"""

from pathlib import Path
import pandas as pd
from collections import defaultdict

# text
TEST_CANCER = "BRCA"
TEST_STATUS = "Normal"
TEST_SAMPLE = "SRR11600329"

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"


def test_microbe_aggregation():
    """textmicrobesreadstext"""
    print("="*80)
    print("text: Sample-level microbe read aggregation")
    print("="*80)
    # textmicrobe list
    csv_path = Path(BASE_CSV) / f"{TEST_CANCER}_V3.csv"
    if not csv_path.exists():
        print(f"Failed CSV file does not exist: {csv_path}")
        return False
    df_microbes = pd.read_csv(csv_path)
    microbes = df_microbes['Taxon'].tolist()
    print(f"OK microbe list: {len(microbes)} ")
    print(f" first5: {microbes[:5]}")
    # textreadtextmicrobesreads
    print(f"\ntextsample: {TEST_CANCER}/{TEST_STATUS}/{TEST_SAMPLE}")
    print("-"*80)
    all_reads = set()
    microbe_contributions = {}
    found_microbes = []
    for i, microbe in enumerate(microbes[:5]): # onlytextfirst5
        txt_path = (Path(BASE_MITRI) / TEST_CANCER / "03.coverage" / TEST_STATUS /             microbe / f"{TEST_SAMPLE}_rna_output.pathseq_sorted.bam.txt")
        if not txt_path.exists():
            print(f" [{i+1}] {microbe}: filetextexists Failed")
            continue
        # textread(text)
        try:
            with open(txt_path, 'r') as f:
                reads_count = 0
                microbe_reads = set()
                for line in f:
                    if line.startswith('@'):
                        continue
                    fields = line.strip().split('\t')
                    if len(fields) >= 10 and fields[9] != '*':
                        read_key = (fields[0], fields[9])
                        microbe_reads.add(read_key)
                        reads_count += 1
                        if reads_count > 0:
                            microbe_contributions[microbe] = reads_count
                            all_reads.update(microbe_reads)
                            found_microbes.append(microbe)
                            print(f" [{i+1}] {microbe}: {reads_count} reads OK")
        except Exception as e:
            print(f" [{i+1}] {microbe}: read error - {e}")
            print(f"\ntextresults:")
            print(f" found {len(found_microbes)} withdatamicrobe")
            print(f" textmicrobereadstextand: {sum(microbe_contributions.values())}")
            print(f" deduplicateafterreadstext: {len(all_reads)}")
            if len(all_reads) > 0:
                overlap_rate = (sum(microbe_contributions.values()) - len(all_reads)) / sum(microbe_contributions.values()) * 100
                print(f" text: {overlap_rate:.2f}% (textmicrobestextreads)")
                return len(found_microbes) > 0


def test_task2_logic():
    """text2text"""
    print("\n" + "="*80)
    print("text: text2textcomparetext")
    print("="*80)
    # textdata
    print("\ntext:")
    print("-"*80)
    # microbe1reads
    microbe1_reads = {
        ('read1', 'ATCG'),
        ('read2', 'GCTA'),
        ('read3', 'TTAA')
    }
    # microbe2reads
    microbe2_reads = {
        ('read2', 'GCTA'), # textmicrobe1text
        ('read4', 'CCGG'),
        ('read5', 'AAAA')
    }
    # microbe3reads
    microbe3_reads = {
        ('read1', 'ATCG'), # textmicrobe1text
        ('read6', 'GGGG')
    }
    print(f"microbe1: {len(microbe1_reads)} reads")
    print(f"microbe2: {len(microbe2_reads)} reads")
    print(f"microbe3: {len(microbe3_reads)} reads")
    # text(sampletextdeduplicate)
    aggregated_mitri = microbe1_reads | microbe2_reads | microbe3_reads
    print(f"\ntextafterMiTRI reads: {len(aggregated_mitri)} uniquereads")
    print(f" textreadstext: {len(microbe1_reads) + len(microbe2_reads) + len(microbe3_reads)}")
    print(f" deduplicateafter: {len(aggregated_mitri)}")
    print(f" text: {len(microbe1_reads) + len(microbe2_reads) + len(microbe3_reads) - len(aggregated_mitri)}")
    # PathSeq reads
    pathseq_reads = {
        ('read1', 'ATCG'), # text
        ('read2', 'GCTA'), # text
        ('read7', 'TTTT'), # PathSeqtextwith
        ('read8', 'CCCC') # PathSeqtextwith
    }
    print(f"\nPathSeq reads: {len(pathseq_reads)}")
    # compare
    intersection = aggregated_mitri & pathseq_reads
    mitri_only = aggregated_mitri - pathseq_reads
    pathseq_only = pathseq_reads - aggregated_mitri
    print(f"\ncompareresults:")
    print(f" text: {len(intersection)} ({len(intersection)/len(aggregated_mitri)*100:.1f}%)")
    print(f" {intersection}")
    print(f" MiTRItextwith: {len(mitri_only)} ({len(mitri_only)/len(aggregated_mitri)*100:.1f}%)")
    print(f" {mitri_only}")
    print(f" PathSeqtextwith: {len(pathseq_only)}")
    print(f" {pathseq_only}")
    # text
    assert len(intersection) == 2, "text2"
    assert len(mitri_only) == 4, "MiTRItextwithtext4"
    print("\nOK textpassed!")
    return True


def test_output_files():
    """textOutput filesformat"""
    print("\n" + "="*80)
    print("text: Output filesformat")
    print("="*80)
    # textgenerateoutput
    sample_name = "TEST_SAMPLE"
    print(f"\ntext2textfortextsamplesgeneratetextfile:")
    print("-"*80)
    print(f"1. {sample_name}_task2_intersection.tsv")
    print(f" - textreadsdetailedinformation")
    print(f" - column: read_id, sequence")
    print()
    print(f"2. {sample_name}_task2_mitri_only.tsv")
    print(f" - textMiTRItextwithreadsdetailedinformation")
    print(f" - column: read_id, sequence")
    print()
    print(f"3. {sample_name}_task2_microbe_contribution.tsv")
    print(f" - recordtextmicrobetextreadstext")
    print(f" - column: microbe, reads_count")
    print()
    # textfileformat
    print("textfile (task2_summary.tsv) column:")
    print("-"*80)
    columns = [
        'cancer', 'status', 'sample', 'num_microbes',
        'total_mitri_reads_aggregated', 'total_pathseq_reads',
        'intersection_count', 'intersection_pct',
        'mitri_only_count', 'mitri_only_pct',
        'max_microbe_reads', 'min_microbe_reads'
    ]
    for i, col in enumerate(columns, 1):
        print(f" {i:2d}. {col}")
        print("\nOK Output filesformattextreasonable")
        return True


def compare_task1_vs_task2():
    """text1andtext2text"""
    print("\n" + "="*80)
    print("text: text1 vs text2")
    print("="*80)
    print("\ntext1 (microbetextcompare):")
    print("-"*80)
    print(" • text: textsamplestextmicrobes")
    print(" • compare: textmicrobesMiTRI reads vs PathSeq reads")
    print(" • output: each(sample, microbe)textrecords")
    print(" • text: textmicrobereadstext")
    print(" • text: BRCA/Normal/SRR001 E.coli textreads")
    print("\ntext2 (sampletextcompare):")
    print("-"*80)
    print(" • text: textsamples")
    print(" • compare: allmicrobetextafterMiTRI reads vs PathSeq reads")
    print(" • output: textsamplestextrecords")
    print(" • text: sampletextmicrobetext")
    print(" • text: BRCA/Normal/SRR001 alltextmicrobetextreads")
    print("\ntext:")
    print("-"*80)
    print(" 1. text:")
    print(" text1: text, textmicrobetext")
    print(" text2: text, deduplicateaftertextcompare")
    print()
    print(" 2. resultscount:")
    print(" text1: sample count × microbetext records")
    print(" text2: sample count records")
    print()
    print(" 3. text:")
    print(" text1: textmicrobetext")
    print(" text2: textsampletextreadstext")
    return True


def main():
    """textrowsalltext"""
    print("\n" + "="*80)
    print("text2textsampletext")
    print("textsample: {}/{}/{}".format(TEST_CANCER, TEST_STATUS, TEST_SAMPLE))
    print("="*80 + "\n")
    results = []
    # textrowstext
    results.append(("microbetext", test_microbe_aggregation()))
    results.append(("text2text", test_task2_logic()))
    results.append(("Output filesformat", test_output_files()))
    results.append(("text1 vs text2text", compare_task1_vs_task2()))
    # text
    print("\n" + "="*80)
    print("text")
    print("="*80)
    for name, result in results:
        status = "OK passed" if result else "Failed failed"
        print(f"{name}: {status}")
        all_passed = all(r[1] for r in results)
        if all_passed:
            print("\n🎉 alltextpassed! textrowstext2text.")
            print("\ntextrowstext:")
            print(" nohup python reads_comparison_task2.py > task2_run.log 2>&1 &")
        else:
            print("\nWarning️ textfailed, textchecktext.")
            print("="*80 + "\n")


if __name__ == "__main__":
    main()

### Main script

In [ ]:
#!/usr/bin/env python3
"""
Task 2: sample-level MiTRI and PathSeq read intersection analysis
For each sample, aggregate MiTRI reads across all microbes and compare them with PathSeq reads
"""

import os
import re
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, Tuple, List
import pandas as pd
import pysam
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75 # reserve 5 of 80 cores for the system

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
OUTPUT_BASE = "/path/to/project/results/summary_V2/reads_compare"


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_task2")
    logger.setLevel(logging.INFO)
    # clear existing handlers
    if logger.handlers:
        logger.handlers.clear()
        # file handler
        fh = logging.FileHandler(output_dir / f"{cancer}_task2_processing.log")
        fh.setLevel(logging.INFO)
        # format
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_pathseq_reads(bam_path: str) -> Set[Tuple[str, str]]:
    """
    Extract reads with YP tags from a PathSeq BAM file
    Task 2 does not need taxids, only the read set
    Returns:
    reads_set: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bam:
            for read in bam:
                if read.has_tag('YP'):
                    read_id = read.query_name
                    sequence = read.query_sequence
                    if sequence is None:
                        continue
                    reads_set.add((read_id, sequence))
    except Exception as e:
        logging.error(f"Error reading PathSeq BAM {bam_path}: {e}")
        return reads_set


def extract_mitri_reads(txt_path: str) -> Set[Tuple[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files
    Returns:
    reads_set: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                if sequence != '*':
                    reads_set.add((read_id, sequence))
    except FileNotFoundError:
        pass # filetextexiststextskip
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_set


def compare_reads(mitri_reads: Set[Tuple[str, str]],     pathseq_reads: Set[Tuple[str, str]]) -> Dict:
"""
Compare MiTRI and PathSeq reads
Returns:
{
'intersection': set,
'mitri_only': set,
'intersection_count': int,
'intersection_pct': float,
'mitri_only_count': int,
'mitri_only_pct': float,
'total_mitri': int,
'total_pathseq': int
}
"""
intersection = mitri_reads & pathseq_reads
mitri_only = mitri_reads - pathseq_reads
total_mitri = len(mitri_reads)
return {
    'intersection': intersection,
    'mitri_only': mitri_only,
    'intersection_count': len(intersection),
    'intersection_pct': (len(intersection) / total_mitri * 100) if total_mitri > 0 else 0,
    'mitri_only_count': len(mitri_only),
    'mitri_only_pct': (len(mitri_only) / total_mitri * 100) if total_mitri > 0 else 0,
    'total_mitri': total_mitri,
    'total_pathseq': len(pathseq_reads)
}


def process_sample_task2(args):
    """
    text2: processtextsamples
    1. textsampleallmicrobeMiTRI reads
    2. textPathSeq readscompare
    """
    cancer, status, sample, microbes, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"[Task2] Processing {cancer}/{status}/{sample}")
    # PathSeq BAMpath
    pathseq_bam = Path(BASE_PATHSEQ) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.warning(f"PathSeq BAM not found: {pathseq_bam}")
        return None
    # extractPathSeq reads
    pathseq_reads = extract_pathseq_reads(str(pathseq_bam))
    logger.info(f"{sample}: PathSeq reads = {len(pathseq_reads)}")
    # textallmicrobeMiTRI reads
    aggregated_mitri_reads = set()
    microbe_read_counts = {} # recordtextmicrobestextreadstext
    for microbe in microbes:
        mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /             microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
        if not mitri_txt.exists():
            continue
        # extracttextmicrobeMiTRI reads
        microbe_reads = extract_mitri_reads(str(mitri_txt))
        if len(microbe_reads) > 0:
            microbe_read_counts[microbe] = len(microbe_reads)
            # texttosampletext(textdeduplicate)
            aggregated_mitri_reads.update(microbe_reads)
            if len(aggregated_mitri_reads) == 0:
                logger.warning(f"{sample}: No MiTRI reads found")
                return None
            logger.info(f"{sample}: Aggregated MiTRI reads from {len(microbe_read_counts)} microbes = {len(aggregated_mitri_reads)}")
            # comparereads
            comparison = compare_reads(aggregated_mitri_reads, pathseq_reads)
            # savedetailedresults - textreads
            detail_file_inter = output_dir / cancer / f"{sample}_task2_intersection.tsv"
            detail_file_inter.parent.mkdir(parents=True, exist_ok=True)
            with open(detail_file_inter, 'w') as f:
                f.write("read_id\tsequence\n")
                for read_id, seq in comparison['intersection']:
                    f.write(f"{read_id}\t{seq}\n")
                    # savedetailedresults - MiTRItextwithreads
                    detail_file_only = output_dir / cancer / f"{sample}_task2_mitri_only.tsv"
                    with open(detail_file_only, 'w') as f:
                        f.write("read_id\tsequence\n")
                        for read_id, seq in comparison['mitri_only']:
                            f.write(f"{read_id}\t{seq}\n")
                            # savemicrobetextstatistics
                            microbe_contrib_file = output_dir / cancer / f"{sample}_task2_microbe_contribution.tsv"
                            with open(microbe_contrib_file, 'w') as f:
                                f.write("microbe\treads_count\n")
                                for microbe, count in sorted(microbe_read_counts.items(), key=lambda x: x[1], reverse=True):
                                    f.write(f"{microbe}\t{count}\n")
                                    logger.info(f"Completed {cancer}/{status}/{sample}: "
                                        f"Intersection={comparison['intersection_count']} ({comparison['intersection_pct']:.2f}%), "
                                        f"MiTRI_only={comparison['mitri_only_count']} ({comparison['mitri_only_pct']:.2f}%)")
                                    # returntextinformation
                                    return {
                                    'cancer': cancer,
                                    'status': status,
                                    'sample': sample,
                                    'num_microbes': len(microbe_read_counts),
                                    'total_mitri_reads_aggregated': comparison['total_mitri'],
                                    'total_pathseq_reads': comparison['total_pathseq'],
                                    'intersection_count': comparison['intersection_count'],
                                    'intersection_pct': comparison['intersection_pct'],
                                    'mitri_only_count': comparison['mitri_only_count'],
                                    'mitri_only_pct': comparison['mitri_only_pct'],
                                    'max_microbe_reads': max(microbe_read_counts.values()) if microbe_read_counts else 0,
                                    'min_microbe_reads': min(microbe_read_counts.values()) if microbe_read_counts else 0
                                }


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext"""
    bam_dir = Path(BASE_PATHSEQ) / cancer / status / "bam"
    if not bam_dir.exists():
        return []
    samples = []
    for bam_file in bam_dir.glob("*_rna_output.pathseq.bam"):
        sample = bam_file.name.replace("_rna_output.pathseq.bam", "")
        samples.append(sample)
        return samples


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer_task2(cancer: str, output_base: Path):
    """processtextcancer typealldata - text2"""
    cancer_output = output_base / cancer
    cancer_output.mkdir(parents=True, exist_ok=True)
    # textlogger
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== [Task2] Starting processing for {cancer} ===")
    # textmicrobe list
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, output_base, f"{cancer}_task2"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_results = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for result in pool.imap_unordered(process_sample_task2, tasks):
                    if result is not None:
                        all_results.append(result)
                        # savecancer typetext
                        if all_results:
                            df_cancer = pd.DataFrame(all_results)
                            summary_file = cancer_output / f"{cancer}_task2_summary.tsv"
                            df_cancer.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== [Task2] Completed {cancer}: {len(all_results)} samples ===")
                            return all_results


def create_visualizations_task2(df: pd.DataFrame, output_dir: Path):
    """createtext2text"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typetext
        ax1 = axes[0, 0]
        cancer_stats = df.groupby('cancer')['intersection_pct'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average Intersection %')
        ax1.set_title('Task2: Average Intersection % by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRItextwithreadstext
        ax2 = axes[0, 1]
        df['mitri_only_pct'].hist(bins=50, ax=ax2, color='coral', edgecolor='black')
        ax2.set_xlabel('MiTRI Only %')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Task2: Distribution of MiTRI-Only Reads %')
        ax2.axvline(df['mitri_only_pct'].mean(), color='red', linestyle='--',             label=f"Mean: {df['mitri_only_pct'].mean():.1f}%")
        ax2.legend()
        # 3. Statuscompare
        ax3 = axes[0, 2]
        status_stats = df.groupby('status')[['intersection_pct', 'mitri_only_pct']].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['steelblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Percentage')
        ax3.set_title('Task2: Intersection vs MiTRI-Only by Status')
        ax3.legend(['Intersection %', 'MiTRI Only %'])
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. textafterMiTRI readscounttext
        ax4 = axes[1, 0]
        df['total_mitri_reads_aggregated'].hist(bins=50, ax=ax4, color='green', edgecolor='black')
        ax4.set_xlabel('Aggregated MiTRI Reads Count')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Task2: Distribution of Aggregated MiTRI Reads')
        ax4.axvline(df['total_mitri_reads_aggregated'].median(), color='red', linestyle='--',
            label=f"Median: {df['total_mitri_reads_aggregated'].median():.0f}")
        ax4.legend()
        # 5. microbecount vs text
        ax5 = axes[1, 1]
        scatter = ax5.scatter(df['num_microbes'], df['intersection_pct'],             c=df['total_mitri_reads_aggregated'],             cmap='viridis', alpha=0.6, s=30)
        ax5.set_xlabel('Number of Microbes')
        ax5.set_ylabel('Intersection %')
        ax5.set_title('Task2: Microbes Count vs Intersection %')
        plt.colorbar(scatter, ax=ax5, label='Total MiTRI Reads')
        ax5.grid(alpha=0.3)
        # 6. textwithreadstext
        ax6 = axes[1, 2]
        cancer_comparison = df.groupby('cancer')[['intersection_count', 'mitri_only_count']].mean()
        cancer_comparison.plot(kind='bar', ax=ax6, color=['steelblue', 'coral'])
        ax6.set_xlabel('Cancer Type')
        ax6.set_ylabel('Average Read Count')
        ax6.set_title('Task2: Average Intersection vs MiTRI-Only Reads')
        ax6.legend(['Intersection', 'MiTRI Only'])
        ax6.set_xticklabels(ax6.get_xticklabels(), rotation=45, ha='right')
        ax6.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'task2_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Task2 visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "task2_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("TASK 2: sampletextanalyze - statistics report\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df)}\n")
        f.write(f"total cancer types: {df['cancer'].nunique()}\n")
        f.write(f"average microbes per sample: {df['num_microbes'].mean():.1f}\n")
        f.write(f"averagetextMiTRI readstext: {df['total_mitri_reads_aggregated'].mean():.0f}\n")
        f.write(f"averagePathSeq readstext: {df['total_pathseq_reads'].mean():.0f}\n")
        f.write(f"\naveragetext: {df['intersection_pct'].mean():.2f}%\n")
        f.write(f"averageMiTRItextwithtext: {df['mitri_only_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df.groupby('cancer').agg({
            'sample': 'count',
            'num_microbes': 'mean',
            'total_mitri_reads_aggregated': 'mean',
            'intersection_pct': 'mean',
            'mitri_only_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average microbes', 'averageMiTRI_reads', 'averagetext%', 'averagetextwith%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads_aggregated': ['mean', 'median'],
            'intersection_pct': ['mean', 'std'],
            'mitri_only_pct': ['mean', 'std']
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # text
        f.write("[4]text\n")
        f.write("-"*80 + "\n")
        bins = [0, 50, 60, 70, 80, 90, 100]
        labels = ['0-50%', '50-60%', '60-70%', '70-80%', '80-90%', '90-100%']
        df['intersection_bin'] = pd.cut(df['intersection_pct'], bins=bins, labels=labels)
        dist = df['intersection_bin'].value_counts().sort_index()
        for label, count in dist.items():
            pct = count / len(df) * 100
            f.write(f"{label}: {count:4d} sample ({pct:5.1f}%)\n")
            f.write("\n")
            # MiTRItextwithreadstextsample
            f.write("[5]MiTRItextwithreadstextTOP 10sample\n")
            f.write("-"*80 + "\n")
            top_mitri_only = df.nlargest(10, 'mitri_only_pct')[
                ['cancer', 'status', 'sample', 'mitri_only_pct', 'mitri_only_count']
            ]
            f.write(top_mitri_only.to_string(index=False))
            f.write("\n\n")
            # textsample
            f.write("[6]textTOP 10sample\n")
            f.write("-"*80 + "\n")
            top_intersection = df.nlargest(10, 'intersection_pct')[
                ['cancer', 'status', 'sample', 'intersection_pct', 'intersection_count']
            ]
            f.write(top_intersection.to_string(index=False))
            f.write("\n\n")
            f.write("="*80 + "\n")
            f.write("End of report\n")
            f.write("="*80 + "\n")
            print(f"OK Task2 statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"Task 2: sample levelMiTRItextPathSeq readstextanalyze")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_results = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_results = process_cancer_task2(cancer, output_base)
        all_results.extend(cancer_results)
        # savetext
        if all_results:
            df_all = pd.DataFrame(all_results)
            summary_file = output_base / "all_samples_task2_summary.tsv"
            df_all.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_all, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_all)}")
            print(f"total cancer types: {df_all['cancer'].nunique()}")
            print(f"\naveragetext: {df_all['intersection_pct'].mean():.2f}%")
            print(f"averageMiTRItextwithtext: {df_all['mitri_only_pct'].mean():.2f}%")
            print(f"\ntextcancer typeaveragetext:")
            cancer_inter = df_all.groupby('cancer')['intersection_pct'].mean().sort_values()
            for cancer, pct in cancer_inter.items():
                print(f" {cancer:6s}: {pct:5.2f}%")
                print(f"\nNormal vs Tumor:")
                status_inter = df_all.groupby('status')['intersection_pct'].mean()
                for status, pct in status_inter.items():
                    print(f" {status:6s}: {pct:5.2f}%")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations_task2(df_all, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"text2analyzecompleted!")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

### Main script, enhanced

In [ ]:
#!/usr/bin/env python3
"""
Task 2 enhanced version: sample-level MiTRI and PathSeq read intersection analysis
New features: 1. Record sample information
2. Record microbe tax_id and multi-match information
3. Merge mitri_only reads from all samples into one large table
"""

import os
import re
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, Tuple, List
import pandas as pd
import pysam
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
OUTPUT_BASE = "/path/to/project/results/summary_V2/reads_compare_sample_level"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_task2")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_task2_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_pathseq_reads(bam_path: str) -> Set[Tuple[str, str]]:
    """
    Extract reads with YP tags from a PathSeq BAM file
    Returns:
    reads_set: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bam:
            for read in bam:
                if read.has_tag('YP'):
                    read_id = read.query_name
                    sequence = read.query_sequence
                    if sequence is None:
                        continue
                    reads_set.add((read_id, sequence))
    except Exception as e:
        logging.error(f"Error reading PathSeq BAM {bam_path}: {e}")
        return reads_set


def extract_mitri_reads(txt_path: str) -> Set[Tuple[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files
    Returns:
    reads_set: {(read_id, sequence), ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                if sequence != '*':
                    reads_set.add((read_id, sequence))
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_set


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Set, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe sources and taxid information for each read
Returns:
aggregated_reads: {(read_id, sequence), ...}
read_to_microbes: {(read_id, sequence): [microbe1, microbe2, ...], ...}
read_to_taxids: {(read_id, sequence): [taxid1, taxid2, ...], ...}
"""
aggregated_reads = set()
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_key in microbe_reads:
            aggregated_reads.add(read_key)
            read_to_microbes[read_key].append(microbe)
            # textmicrobealltaxids
            for taxid in microbe_taxids:
                read_to_taxids[read_key].add(taxid)
                # texttaxidsforlisttextaftertextprocess
                read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                return aggregated_reads, dict(read_to_microbes), read_to_taxids


def compare_reads_enhanced(mitri_reads: Set[Tuple[str, str]],     pathseq_reads: Set[Tuple[str, str]],
    read_to_microbes: Dict,
    read_to_taxids: Dict) -> Dict:
"""
textreadscompare, returndetailedinformation
"""
intersection = mitri_reads & pathseq_reads
mitri_only = mitri_reads - pathseq_reads
total_mitri = len(mitri_reads)
return {
    'intersection': intersection,
    'mitri_only': mitri_only,
    'intersection_count': len(intersection),
    'intersection_pct': (len(intersection) / total_mitri * 100) if total_mitri > 0 else 0,
    'mitri_only_count': len(mitri_only),
    'mitri_only_pct': (len(mitri_only) / total_mitri * 100) if total_mitri > 0 else 0,
    'total_mitri': total_mitri,
    'total_pathseq': len(pathseq_reads),
    'read_to_microbes': read_to_microbes,
    'read_to_taxids': read_to_taxids
}


def save_detailed_results(cancer: str, status: str, sample: str,     comparison: Dict, output_dir: Path):
"""
savedetailedresults, textsample, microbeandtaxidinformation
"""
output_cancer_dir = output_dir / cancer
output_cancer_dir.mkdir(parents=True, exist_ok=True)
read_to_microbes = comparison['read_to_microbes']
read_to_taxids = comparison['read_to_taxids']
# 1. savetextreads
intersection_records = []
for read_id, seq in comparison['intersection']:
    microbes = read_to_microbes.get((read_id, seq), [])
    taxids = read_to_taxids.get((read_id, seq), [])
    intersection_records.append({
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'read_id': read_id,
        'sequence': seq,
        'microbes_used': '|'.join(microbes),
        'num_microbes': len(microbes),
        'taxids': '|'.join(map(str, taxids)),
        'num_taxids': len(taxids)
    })
    if intersection_records:
        df_inter = pd.DataFrame(intersection_records)
        inter_file = output_cancer_dir / f"{sample}_task2_intersection.tsv"
        df_inter.to_csv(inter_file, sep='\t', index=False)
        # 2. Save MiTRI-only reads(text)
        mitri_only_records = []
        for read_id, seq in comparison['mitri_only']:
            microbes = read_to_microbes.get((read_id, seq), [])
            taxids = read_to_taxids.get((read_id, seq), [])
            mitri_only_records.append({
                'cancer': cancer,
                'status': status,
                'sample': sample,
                'read_id': read_id,
                'sequence': seq,
                'microbes_used': '|'.join(microbes),
                'num_microbes': len(microbes),
                'taxids': '|'.join(map(str, taxids)),
                'num_taxids': len(taxids)
            })
            if mitri_only_records:
                df_only = pd.DataFrame(mitri_only_records)
                only_file = output_cancer_dir / f"{sample}_task2_mitri_only.tsv"
                df_only.to_csv(only_file, sep='\t', index=False)
                return mitri_only_records # returntextaftertext


def process_sample_task2(args):
    """
    text2text: processtextsamples
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"[Task2] Processing {cancer}/{status}/{sample}")
    # PathSeq BAMpath
    pathseq_bam = Path(BASE_PATHSEQ) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.warning(f"PathSeq BAM not found: {pathseq_bam}")
        return None, []
    # extractPathSeq reads
    pathseq_reads = extract_pathseq_reads(str(pathseq_bam))
    logger.info(f"{sample}: PathSeq reads = {len(pathseq_reads)}")
    # textMiTRI reads(textdetailedinformation)
    aggregated_reads, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(aggregated_reads) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None, []
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Aggregated MiTRI reads from {num_microbes_used} microbes = {len(aggregated_reads)}")
    # comparereads
    comparison = compare_reads_enhanced(aggregated_reads, pathseq_reads,
        read_to_microbes, read_to_taxids)
    # savedetailedresultstextmitri_onlyrecord
    mitri_only_records = save_detailed_results(cancer, status, sample, comparison, output_dir)
    logger.info(f"Completed {cancer}/{status}/{sample}: "
        f"Intersection={comparison['intersection_count']} ({comparison['intersection_pct']:.2f}%), "
        f"MiTRI_only={comparison['mitri_only_count']} ({comparison['mitri_only_pct']:.2f}%)")
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'num_microbes': num_microbes_used,
        'total_mitri_reads_aggregated': comparison['total_mitri'],
        'total_pathseq_reads': comparison['total_pathseq'],
        'intersection_count': comparison['intersection_count'],
        'intersection_pct': comparison['intersection_pct'],
        'mitri_only_count': comparison['mitri_only_count'],
        'mitri_only_pct': comparison['mitri_only_pct']
    }
    return summary, mitri_only_records


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext"""
    bam_dir = Path(BASE_PATHSEQ) / cancer / status / "bam"
    if not bam_dir.exists():
        return []
    samples = []
    for bam_file in bam_dir.glob("*_rna_output.pathseq.bam"):
        sample = bam_file.name.replace("_rna_output.pathseq.bam", "")
        samples.append(sample)
        return samples


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer_task2(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer type - text2text"""
    cancer_output = output_base / cancer
    cancer_output.mkdir(parents=True, exist_ok=True)
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== [Task2] Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_task2"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            all_mitri_only_records = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary, mitri_only_recs in pool.imap_unordered(process_sample_task2, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        all_mitri_only_records.extend(mitri_only_recs)
                        # savecancer typetext
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = cancer_output / f"{cancer}_task2_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            # savecancer typetextallmitri_only reads
                            if all_mitri_only_records:
                                df_mitri_only = pd.DataFrame(all_mitri_only_records)
                                mitri_only_file = cancer_output / f"{cancer}_all_samples_mitri_only.tsv"
                                df_mitri_only.to_csv(mitri_only_file, sep='\t', index=False)
                                logger.info(f"Saved {len(all_mitri_only_records)} mitri_only reads for {cancer}")
                                logger.info(f"=== [Task2] Completed {cancer}: {len(all_summaries)} samples ===")
                                return all_summaries, all_mitri_only_records


def create_visualizations_task2(df: pd.DataFrame, output_dir: Path):
    """createtext2text"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typetext
        ax1 = axes[0, 0]
        cancer_stats = df.groupby('cancer')['intersection_pct'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average Intersection %')
        ax1.set_title('Task2: Average Intersection % by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRItextwithreadstext
        ax2 = axes[0, 1]
        df['mitri_only_pct'].hist(bins=50, ax=ax2, color='coral', edgecolor='black')
        ax2.set_xlabel('MiTRI Only %')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Task2: Distribution of MiTRI-Only Reads %')
        ax2.axvline(df['mitri_only_pct'].mean(), color='red', linestyle='--',             label=f"Mean: {df['mitri_only_pct'].mean():.1f}%")
        ax2.legend()
        # 3. Statuscompare
        ax3 = axes[0, 2]
        status_stats = df.groupby('status')[['intersection_pct', 'mitri_only_pct']].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['steelblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Percentage')
        ax3.set_title('Task2: Intersection vs MiTRI-Only by Status')
        ax3.legend(['Intersection %', 'MiTRI Only %'])
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. textafterMiTRI readscounttext
        ax4 = axes[1, 0]
        df['total_mitri_reads_aggregated'].hist(bins=50, ax=ax4, color='green', edgecolor='black')
        ax4.set_xlabel('Aggregated MiTRI Reads Count')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Task2: Distribution of Aggregated MiTRI Reads')
        ax4.axvline(df['total_mitri_reads_aggregated'].median(), color='red', linestyle='--',
            label=f"Median: {df['total_mitri_reads_aggregated'].median():.0f}")
        ax4.legend()
        # 5. microbecount vs text
        ax5 = axes[1, 1]
        scatter = ax5.scatter(df['num_microbes'], df['intersection_pct'],             c=df['total_mitri_reads_aggregated'],             cmap='viridis', alpha=0.6, s=30)
        ax5.set_xlabel('Number of Microbes')
        ax5.set_ylabel('Intersection %')
        ax5.set_title('Task2: Microbes Count vs Intersection %')
        plt.colorbar(scatter, ax=ax5, label='Total MiTRI Reads')
        ax5.grid(alpha=0.3)
        # 6. textwithreadstext
        ax6 = axes[1, 2]
        cancer_comparison = df.groupby('cancer')[['intersection_count', 'mitri_only_count']].mean()
        cancer_comparison.plot(kind='bar', ax=ax6, color=['steelblue', 'coral'])
        ax6.set_xlabel('Cancer Type')
        ax6.set_ylabel('Average Read Count')
        ax6.set_title('Task2: Average Intersection vs MiTRI-Only Reads')
        ax6.legend(['Intersection', 'MiTRI Only'])
        ax6.set_xticklabels(ax6.get_xticklabels(), rotation=45, ha='right')
        ax6.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'task2_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Task2 visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, df_mitri_only: pd.DataFrame,     output_dir: Path):
"""generatedetailedstatistics report(textmitri_onlydetailedstatistics)"""
report_file = output_dir / "task2_statistics_report.txt"
with open(report_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TASK 2: sampletextanalyze - statistics report(text)\n")
    f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*80 + "\n\n")
    # Overall statistics
    f.write("[1]Overall statistics\n")
    f.write("-"*80 + "\n")
    f.write(f"textsample count: {len(df_summary)}\n")
    f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
    f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
    f.write(f"averagetextMiTRI readstext: {df_summary['total_mitri_reads_aggregated'].mean():.0f}\n")
    f.write(f"averagePathSeq readstext: {df_summary['total_pathseq_reads'].mean():.0f}\n")
    f.write(f"\naveragetext: {df_summary['intersection_pct'].mean():.2f}%\n")
    f.write(f"averageMiTRItextwithtext: {df_summary['mitri_only_pct'].mean():.2f}%\n")
    f.write("\n")
    # MiTRI onlydetailedstatistics
    f.write("[2]MiTRItextwithreadsdetailedstatistics\n")
    f.write("-"*80 + "\n")
    f.write(f"textMiTRItextwithreadstext: {len(df_mitri_only)}\n")
    f.write(f"textsample count: {df_mitri_only['sample'].nunique()}\n")
    f.write(f"textcancer typetext: {df_mitri_only['cancer'].nunique()}\n")
    f.write(f"\naverage microbes using each read: {df_mitri_only['num_microbes'].mean():.2f}\n")
    f.write(f"averageeachreadtexttaxidtext: {df_mitri_only['num_taxids'].mean():.2f}\n")
    # textmatchedstatistics
    multi_match = df_mitri_only[df_mitri_only['num_microbes'] > 1]
    f.write(f"\ntextmatchedreads (text>1microbestext): {len(multi_match)} ({len(multi_match)/len(df_mitri_only)*100:.2f}%)\n")
    f.write(f" text {df_mitri_only['num_microbes'].max()} microbestext\n")
    f.write("\n")
    # Summarize by cancer type
    f.write("[3]statistics by cancer type\n")
    f.write("-"*80 + "\n")
    cancer_stats = df_summary.groupby('cancer').agg({
        'sample': 'count',
        'num_microbes': 'mean',
        'total_mitri_reads_aggregated': 'mean',
        'intersection_pct': 'mean',
        'mitri_only_pct': 'mean'
    }).round(2)
    cancer_stats.columns = ['sample count', 'average microbes', 'averageMiTRI_reads', 'averagetext%', 'averagetextwith%']
    f.write(cancer_stats.to_string())
    # eachcancer typemitri_onlystatistics
    f.write("\n\ntextcancer typeMiTRItextwithreadsstatistics:\n")
    cancer_mitri_only = df_mitri_only.groupby('cancer').agg({
        'read_id': 'count',
        'num_microbes': 'mean',
        'num_taxids': 'mean'
    }).round(2)
    cancer_mitri_only.columns = ['textwithreadstext', 'average microbes/read', 'averagetaxidtext/read']
    f.write(cancer_mitri_only.to_string())
    f.write("\n\n")
    # textstatistics
    f.write("[4]Normal vs Tumor statistics\n")
    f.write("-"*80 + "\n")
    status_stats = df_summary.groupby('status').agg({
        'sample': 'count',
        'total_mitri_reads_aggregated': ['mean', 'median'],
        'intersection_pct': ['mean', 'std'],
        'mitri_only_pct': ['mean', 'std']
    }).round(2)
    f.write(status_stats.to_string())
    f.write("\n\n")
    f.write("="*80 + "\n")
    f.write("End of report\n")
    f.write("="*80 + "\n")
    print(f"OK Task2 enhanced statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"text2text: sampletextMiTRItextPathSeq readstextanalyze")
    print(f"text: recordsample, microbetaxid, textmatchedinformation, textallresults")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    all_mitri_only_records = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries, cancer_mitri_only = process_cancer_task2(
            cancer, output_base, microbe_to_taxids
)
all_summaries.extend(cancer_summaries)
all_mitri_only_records.extend(cancer_mitri_only)
# savetext
if all_summaries:
    df_summary = pd.DataFrame(all_summaries)
    summary_file = output_base / "all_samples_task2_summary.tsv"
    df_summary.to_csv(summary_file, sep='\t', index=False)
    print(f"\nOK Saved global summary: {summary_file}")
    # [textnew]savetextmitri_onlytext
    if all_mitri_only_records:
        df_mitri_only_all = pd.DataFrame(all_mitri_only_records)
        mitri_only_all_file = output_base / "all_cancers_all_samples_mitri_only.tsv"
        df_mitri_only_all.to_csv(mitri_only_all_file, sep='\t', index=False)
        print(f"OK Saved merged mitri_only table: {mitri_only_all_file}")
        print(f" Total mitri_only reads: {len(df_mitri_only_all)}")
        print(f" Unique samples: {df_mitri_only_all['sample'].nunique()}")
        print(f" Unique cancers: {df_mitri_only_all['cancer'].nunique()}")
        # generatetextstatistics report
        if all_summaries and all_mitri_only_records:
            print("\n" + "="*80)
            print("generatetextstatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, df_mitri_only_all, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetext: {df_summary['intersection_pct'].mean():.2f}%")
            print(f"averageMiTRItextwithtext: {df_summary['mitri_only_pct'].mean():.2f}%")
            print(f"\n[MiTRItextwithreadstext]")
            print(f"text: {len(df_mitri_only_all)}")
            print(f"averagetextreadtext{df_mitri_only_all['num_microbes'].mean():.2f}microbestext")
            multi_match = df_mitri_only_all[df_mitri_only_all['num_microbes'] > 1]
            print(f"textmatchedreads: {len(multi_match)} ({len(multi_match)/len(df_mitri_only_all)*100:.1f}%)")
            # createtext
            print("\n" + "="*80)
            print("generatetext...")
            print("="*80)
            create_visualizations_task2(df_summary, output_base)
            elapsed_time = time.time() - start_time
            print(f"\n{'='*80}")
            print(f"text2textanalyzecompleted!")
            print(f"text: {elapsed_time/60:.2f} text")
            print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

### Main script, read IDs only

In [ ]:
#!/usr/bin/env python3
"""
Task 2 enhanced version: sample-level MiTRI and PathSeq read intersection analysis
text: Compare only by read_id and do not save sequences
New features: 1. Record sample information
2. Record microbe tax_id and multi-match information
3. Merge mitri_only reads from all samples into one large table
"""

import os
import re
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List
import pandas as pd
import pysam
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75

# Path configuration
BASE_PATHSEQ = "/path/to/project/data/cancers_V1"
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
OUTPUT_BASE = "/path/to/project/results/summary_V2/reads_compare_sample_level_readsID"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_task2")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_task2_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_pathseq_reads(bam_path: str) -> Set[str]:
    """
    Extract reads with YP tags from a PathSeq BAM file(textread_id)
    Returns:
    reads_set: {read_id1, read_id2, ...}
    """
    reads_set = set()
    try:
        with pysam.AlignmentFile(bam_path, "rb") as bam:
            for read in bam:
                if read.has_tag('YP'):
                    read_id = read.query_name
                    reads_set.add(read_id)
    except Exception as e:
        logging.error(f"Error reading PathSeq BAM {bam_path}: {e}")
        return reads_set


def extract_mitri_reads(txt_path: str) -> Set[str]:
    """
    Extract reads from MiTRI BAM.txt files(textread_id)
    Returns:
    reads_set: {read_id1, read_id2, ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_set.add(read_id)
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_set


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> tuple:
"""
Aggregate sample MiTRI reads and record microbe sources and taxid information for each read
Returns:
aggregated_reads: {read_id1, read_id2, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
aggregated_reads = set()
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id in microbe_reads:
            aggregated_reads.add(read_id)
            read_to_microbes[read_id].append(microbe)
            # textmicrobealltaxids
            for taxid in microbe_taxids:
                read_to_taxids[read_id].add(taxid)
                # texttaxidsforlisttextaftertextprocess
                read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                return aggregated_reads, dict(read_to_microbes), read_to_taxids


def compare_reads_enhanced(mitri_reads: Set[str],     pathseq_reads: Set[str],
    read_to_microbes: Dict,
    read_to_taxids: Dict) -> Dict:
"""
textreadscompare, returndetailedinformation
"""
intersection = mitri_reads & pathseq_reads
mitri_only = mitri_reads - pathseq_reads
total_mitri = len(mitri_reads)
return {
    'intersection': intersection,
    'mitri_only': mitri_only,
    'intersection_count': len(intersection),
    'intersection_pct': (len(intersection) / total_mitri * 100) if total_mitri > 0 else 0,
    'mitri_only_count': len(mitri_only),
    'mitri_only_pct': (len(mitri_only) / total_mitri * 100) if total_mitri > 0 else 0,
    'total_mitri': total_mitri,
    'total_pathseq': len(pathseq_reads),
    'read_to_microbes': read_to_microbes,
    'read_to_taxids': read_to_taxids
}


def save_detailed_results(cancer: str, status: str, sample: str,     comparison: Dict, output_dir: Path):
"""
savedetailedresults, textsample, microbeandtaxidinformation(textsavesequence)
"""
output_cancer_dir = output_dir / cancer
output_cancer_dir.mkdir(parents=True, exist_ok=True)
read_to_microbes = comparison['read_to_microbes']
read_to_taxids = comparison['read_to_taxids']
# 1. savetextreads
intersection_records = []
for read_id in comparison['intersection']:
    microbes = read_to_microbes.get(read_id, [])
    taxids = read_to_taxids.get(read_id, [])
    intersection_records.append({
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'read_id': read_id,
        'microbes_used': '|'.join(microbes),
        'num_microbes': len(microbes),
        'taxids': '|'.join(map(str, taxids)),
        'num_taxids': len(taxids)
    })
    if intersection_records:
        df_inter = pd.DataFrame(intersection_records)
        inter_file = output_cancer_dir / f"{sample}_task2_intersection.tsv"
        df_inter.to_csv(inter_file, sep='\t', index=False)
        # 2. Save MiTRI-only reads(text)
        mitri_only_records = []
        for read_id in comparison['mitri_only']:
            microbes = read_to_microbes.get(read_id, [])
            taxids = read_to_taxids.get(read_id, [])
            mitri_only_records.append({
                'cancer': cancer,
                'status': status,
                'sample': sample,
                'read_id': read_id,
                'microbes_used': '|'.join(microbes),
                'num_microbes': len(microbes),
                'taxids': '|'.join(map(str, taxids)),
                'num_taxids': len(taxids)
            })
            if mitri_only_records:
                df_only = pd.DataFrame(mitri_only_records)
                only_file = output_cancer_dir / f"{sample}_task2_mitri_only.tsv"
                df_only.to_csv(only_file, sep='\t', index=False)
                return mitri_only_records # returntextaftertext


def process_sample_task2(args):
    """
    text2text: processtextsamples
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"[Task2] Processing {cancer}/{status}/{sample}")
    # PathSeq BAMpath
    pathseq_bam = Path(BASE_PATHSEQ) / cancer / status / "bam" / f"{sample}_rna_output.pathseq.bam"
    if not pathseq_bam.exists():
        logger.warning(f"PathSeq BAM not found: {pathseq_bam}")
        return None, []
    # extractPathSeq reads
    pathseq_reads = extract_pathseq_reads(str(pathseq_bam))
    logger.info(f"{sample}: PathSeq reads = {len(pathseq_reads)}")
    # textMiTRI reads(textdetailedinformation)
    aggregated_reads, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(aggregated_reads) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None, []
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Aggregated MiTRI reads from {num_microbes_used} microbes = {len(aggregated_reads)}")
    # comparereads
    comparison = compare_reads_enhanced(aggregated_reads, pathseq_reads,
        read_to_microbes, read_to_taxids)
    # savedetailedresultstextmitri_onlyrecord
    mitri_only_records = save_detailed_results(cancer, status, sample, comparison, output_dir)
    logger.info(f"Completed {cancer}/{status}/{sample}: "
        f"Intersection={comparison['intersection_count']} ({comparison['intersection_pct']:.2f}%), "
        f"MiTRI_only={comparison['mitri_only_count']} ({comparison['mitri_only_pct']:.2f}%)")
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'num_microbes': num_microbes_used,
        'total_mitri_reads_aggregated': comparison['total_mitri'],
        'total_pathseq_reads': comparison['total_pathseq'],
        'intersection_count': comparison['intersection_count'],
        'intersection_pct': comparison['intersection_pct'],
        'mitri_only_count': comparison['mitri_only_count'],
        'mitri_only_pct': comparison['mitri_only_pct']
    }
    return summary, mitri_only_records


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext"""
    bam_dir = Path(BASE_PATHSEQ) / cancer / status / "bam"
    if not bam_dir.exists():
        return []
    samples = []
    for bam_file in bam_dir.glob("*_rna_output.pathseq.bam"):
        sample = bam_file.name.replace("_rna_output.pathseq.bam", "")
        samples.append(sample)
        return samples


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer_task2(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer type - text2text"""
    cancer_output = output_base / cancer
    cancer_output.mkdir(parents=True, exist_ok=True)
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== [Task2] Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_task2"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            all_mitri_only_records = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary, mitri_only_recs in pool.imap_unordered(process_sample_task2, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        all_mitri_only_records.extend(mitri_only_recs)
                        # savecancer typetext
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = cancer_output / f"{cancer}_task2_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            # savecancer typetextallmitri_only reads
                            if all_mitri_only_records:
                                df_mitri_only = pd.DataFrame(all_mitri_only_records)
                                mitri_only_file = cancer_output / f"{cancer}_all_samples_mitri_only.tsv"
                                df_mitri_only.to_csv(mitri_only_file, sep='\t', index=False)
                                logger.info(f"Saved {len(all_mitri_only_records)} mitri_only reads for {cancer}")
                                logger.info(f"=== [Task2] Completed {cancer}: {len(all_summaries)} samples ===")
                                return all_summaries, all_mitri_only_records


def create_visualizations_task2(df: pd.DataFrame, output_dir: Path):
    """createtext2text"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typetext
        ax1 = axes[0, 0]
        cancer_stats = df.groupby('cancer')['intersection_pct'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average Intersection %')
        ax1.set_title('Task2: Average Intersection % by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRItextwithreadstext
        ax2 = axes[0, 1]
        df['mitri_only_pct'].hist(bins=50, ax=ax2, color='coral', edgecolor='black')
        ax2.set_xlabel('MiTRI Only %')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Task2: Distribution of MiTRI-Only Reads %')
        ax2.axvline(df['mitri_only_pct'].mean(), color='red', linestyle='--',             label=f"Mean: {df['mitri_only_pct'].mean():.1f}%")
        ax2.legend()
        # 3. Statuscompare
        ax3 = axes[0, 2]
        status_stats = df.groupby('status')[['intersection_pct', 'mitri_only_pct']].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['steelblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Percentage')
        ax3.set_title('Task2: Intersection vs MiTRI-Only by Status')
        ax3.legend(['Intersection %', 'MiTRI Only %'])
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. textafterMiTRI readscounttext
        ax4 = axes[1, 0]
        df['total_mitri_reads_aggregated'].hist(bins=50, ax=ax4, color='green', edgecolor='black')
        ax4.set_xlabel('Aggregated MiTRI Reads Count')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Task2: Distribution of Aggregated MiTRI Reads')
        ax4.axvline(df['total_mitri_reads_aggregated'].median(), color='red', linestyle='--',
            label=f"Median: {df['total_mitri_reads_aggregated'].median():.0f}")
        ax4.legend()
        # 5. microbecount vs text
        ax5 = axes[1, 1]
        scatter = ax5.scatter(df['num_microbes'], df['intersection_pct'],             c=df['total_mitri_reads_aggregated'],             cmap='viridis', alpha=0.6, s=30)
        ax5.set_xlabel('Number of Microbes')
        ax5.set_ylabel('Intersection %')
        ax5.set_title('Task2: Microbes Count vs Intersection %')
        plt.colorbar(scatter, ax=ax5, label='Total MiTRI Reads')
        ax5.grid(alpha=0.3)
        # 6. textwithreadstext
        ax6 = axes[1, 2]
        cancer_comparison = df.groupby('cancer')[['intersection_count', 'mitri_only_count']].mean()
        cancer_comparison.plot(kind='bar', ax=ax6, color=['steelblue', 'coral'])
        ax6.set_xlabel('Cancer Type')
        ax6.set_ylabel('Average Read Count')
        ax6.set_title('Task2: Average Intersection vs MiTRI-Only Reads')
        ax6.legend(['Intersection', 'MiTRI Only'])
        ax6.set_xticklabels(ax6.get_xticklabels(), rotation=45, ha='right')
        ax6.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'task2_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Task2 visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, df_mitri_only: pd.DataFrame,     output_dir: Path):
"""generatedetailedstatistics report(textmitri_onlydetailedstatistics)"""
report_file = output_dir / "task2_statistics_report.txt"
with open(report_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("TASK 2: sampletextanalyze - statistics report(text: textread_idcompare)\n")
    f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*80 + "\n\n")
    # Overall statistics
    f.write("[1]Overall statistics\n")
    f.write("-"*80 + "\n")
    f.write(f"textsample count: {len(df_summary)}\n")
    f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
    f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
    f.write(f"averagetextMiTRI readstext: {df_summary['total_mitri_reads_aggregated'].mean():.0f}\n")
    f.write(f"averagePathSeq readstext: {df_summary['total_pathseq_reads'].mean():.0f}\n")
    f.write(f"\naveragetext: {df_summary['intersection_pct'].mean():.2f}%\n")
    f.write(f"averageMiTRItextwithtext: {df_summary['mitri_only_pct'].mean():.2f}%\n")
    f.write("\n")
    # MiTRI onlydetailedstatistics
    f.write("[2]MiTRItextwithreadsdetailedstatistics\n")
    f.write("-"*80 + "\n")
    f.write(f"textMiTRItextwithreadstext: {len(df_mitri_only)}\n")
    f.write(f"textsample count: {df_mitri_only['sample'].nunique()}\n")
    f.write(f"textcancer typetext: {df_mitri_only['cancer'].nunique()}\n")
    f.write(f"\naverage microbes using each read: {df_mitri_only['num_microbes'].mean():.2f}\n")
    f.write(f"averageeachreadtexttaxidtext: {df_mitri_only['num_taxids'].mean():.2f}\n")
    # textmatchedstatistics
    multi_match = df_mitri_only[df_mitri_only['num_microbes'] > 1]
    f.write(f"\ntextmatchedreads (text>1microbestext): {len(multi_match)} ({len(multi_match)/len(df_mitri_only)*100:.2f}%)\n")
    f.write(f" text {df_mitri_only['num_microbes'].max()} microbestext\n")
    f.write("\n")
    # Summarize by cancer type
    f.write("[3]statistics by cancer type\n")
    f.write("-"*80 + "\n")
    cancer_stats = df_summary.groupby('cancer').agg({
        'sample': 'count',
        'num_microbes': 'mean',
        'total_mitri_reads_aggregated': 'mean',
        'intersection_pct': 'mean',
        'mitri_only_pct': 'mean'
    }).round(2)
    cancer_stats.columns = ['sample count', 'average microbes', 'averageMiTRI_reads', 'averagetext%', 'averagetextwith%']
    f.write(cancer_stats.to_string())
    # eachcancer typemitri_onlystatistics
    f.write("\n\ntextcancer typeMiTRItextwithreadsstatistics:\n")
    cancer_mitri_only = df_mitri_only.groupby('cancer').agg({
        'read_id': 'count',
        'num_microbes': 'mean',
        'num_taxids': 'mean'
    }).round(2)
    cancer_mitri_only.columns = ['textwithreadstext', 'average microbes/read', 'averagetaxidtext/read']
    f.write(cancer_mitri_only.to_string())
    f.write("\n\n")
    # textstatistics
    f.write("[4]Normal vs Tumor statistics\n")
    f.write("-"*80 + "\n")
    status_stats = df_summary.groupby('status').agg({
        'sample': 'count',
        'total_mitri_reads_aggregated': ['mean', 'median'],
        'intersection_pct': ['mean', 'std'],
        'mitri_only_pct': ['mean', 'std']
    }).round(2)
    f.write(status_stats.to_string())
    f.write("\n\n")
    f.write("="*80 + "\n")
    f.write("End of report\n")
    f.write("="*80 + "\n")
    print(f"OK Task2 statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"text2text: sampletextMiTRItextPathSeq readstextanalyze(textread_idcompare)")
    print(f"text: recordsample, microbetaxid, textmatchedinformation, textallresults")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    all_mitri_only_records = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries, cancer_mitri_only = process_cancer_task2(
            cancer, output_base, microbe_to_taxids
)
all_summaries.extend(cancer_summaries)
all_mitri_only_records.extend(cancer_mitri_only)
# savetext
if all_summaries:
    df_summary = pd.DataFrame(all_summaries)
    summary_file = output_base / "all_samples_task2_summary.tsv"
    df_summary.to_csv(summary_file, sep='\t', index=False)
    print(f"\nOK Saved global summary: {summary_file}")
    # [textnew]savetextmitri_onlytext
    if all_mitri_only_records:
        df_mitri_only_all = pd.DataFrame(all_mitri_only_records)
        mitri_only_all_file = output_base / "all_cancers_all_samples_mitri_only.tsv"
        df_mitri_only_all.to_csv(mitri_only_all_file, sep='\t', index=False)
        print(f"OK Saved merged mitri_only table: {mitri_only_all_file}")
        print(f" Total mitri_only reads: {len(df_mitri_only_all)}")
        print(f" Unique read_ids: {df_mitri_only_all['read_id'].nunique()}")
        print(f" Unique samples: {df_mitri_only_all['sample'].nunique()}")
        print(f" Unique cancers: {df_mitri_only_all['cancer'].nunique()}")
        # generatetextstatistics report
        if all_summaries and all_mitri_only_records:
            print("\n" + "="*80)
            print("generatetextstatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, df_mitri_only_all, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetext: {df_summary['intersection_pct'].mean():.2f}%")
            print(f"averageMiTRItextwithtext: {df_summary['mitri_only_pct'].mean():.2f}%")
            print(f"\n[MiTRItextwithreadstext]")
            print(f"total records: {len(df_mitri_only_all)}")
            print(f"uniqueread_idtext: {df_mitri_only_all['read_id'].nunique()}")
            print(f"averagetextreadtext{df_mitri_only_all['num_microbes'].mean():.2f}microbestext")
            multi_match = df_mitri_only_all[df_mitri_only_all['num_microbes'] > 1]
            print(f"textmatchedreadsrecord: {len(multi_match)} ({len(multi_match)/len(df_mitri_only_all)*100:.1f}%)")
            # createtext
            print("\n" + "="*80)
            print("generatetext...")
            print("="*80)
            create_visualizations_task2(df_summary, output_base)
            elapsed_time = time.time() - start_time
            print(f"\n{'='*80}")
            print(f"text2textanalyzecompleted!")
            print(f"text: {elapsed_time/60:.2f} text")
            print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
            print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

### Diagnose sample differences between two result sets

In [ ]:
#!/usr/bin/env python3
"""Diagnose sample differences between two result sets"""

import pandas as pd

# Read the two summary files
df1 = pd.read_csv('/path/to/project/results/summary_V2/reads_compare_sample_level/all_samples_task2_summary.tsv', sep='\t')
df2 = pd.read_csv('/path/to/project/results/summary_V2/reads_compare_sample_level_readsID/all_samples_task2_summary.tsv', sep='\t')

print(f"Script 1 sample count: {len(df1)}")
print(f"Script 2 sample count: {len(df2)}")

# Get sample identifiers
samples1 = set(df1['cancer'] + '_' + df1['status'] + '_' + df1['sample'])
samples2 = set(df2['cancer'] + '_' + df2['status'] + '_' + df2['sample'])

# Find differences
only_in_1 = samples1 - samples2
only_in_2 = samples2 - samples1

print(f"\nSamples only in script 1: {len(only_in_1)}")
if len(only_in_1) <= 10:
    for s in sorted(only_in_1):
        print(f" {s}")
else:
    for s in sorted(list(only_in_1)[:10]):
        print(f" {s}")
        print(f" ... more {len(only_in_1)-10} ")

print(f"\nSamples only in script 2: {len(only_in_2)}")
if len(only_in_2) <= 10:
    for s in sorted(only_in_2):
        print(f" {s}")

# checktextwithMiTRI readsfor0text
print("\n=== MiTRI read statistics ===")
print(f"text1inMiTRI=0sample count: {len(df1[df1['total_mitri_reads_aggregated'] == 0])}")
print(f"text2inMiTRI=0sample count: {len(df2[df2['total_mitri_reads_aggregated'] == 0])}")

# Summarize by cancer type
print("\n=== Sample-count comparison by cancer type ===")
cancer_comp = pd.DataFrame({
    'text1': df1.groupby('cancer')['sample'].count(),
    'text2': df2.groupby('cancer')['sample'].count()
})
cancer_comp['text'] = cancer_comp['text1'] - cancer_comp['text2']
print(cancer_comp)

In [ ]:
#!/usr/bin/env python3
"""Check samples with 100% intersection"""

import pandas as pd

df1 = pd.read_csv('/path/to/project/results/summary_V2/reads_compare_sample_level/all_samples_task2_summary.tsv', sep='\t')
df2 = pd.read_csv('/path/to/project/results/summary_V2/reads_compare_sample_level_readsID/all_samples_task2_summary.tsv', sep='\t')

print("=== Statistics for samples with 100% intersection ===\n")

# text1
full_intersection_1 = df1[df1['intersection_pct'] == 100]
print(f"text1: {len(full_intersection_1)} samplestext100%")
print(f" samples with mitri_only reads: {len(df1[df1['mitri_only_count'] > 0])}")
print(f" samples without mitri_only reads: {len(df1[df1['mitri_only_count'] == 0])}")

print()

# text2
full_intersection_2 = df2[df2['intersection_pct'] == 100]
print(f"text2: {len(full_intersection_2)} samplestext100%")
print(f" samples with mitri_only reads: {len(df2[df2['mitri_only_count'] > 0])}")
print(f" samples without mitri_only reads: {len(df2[df2['mitri_only_count'] == 0])}")

print("\n=== Why the two scripts produce different results? ===")
print("Script 1 compares using the (read_id, sequence) pair")
print("Script 2 compares using read_id only")
print("\nPossible reason: ")
print("1. Mixed single-end and paired-end data: the same read_id may correspond to different sequences (R1/R2)")
print("2. text1textR1andR2, text2textfortextread")
print("3. text2text(textsampletextto100%)")

# text
print("\n=== text100%sampletext(text2)===")
if len(full_intersection_2) > 0:
    print(full_intersection_2[['cancer', 'status', 'sample', 'total_mitri_reads_aggregated',         'intersection_count', 'mitri_only_count']].head(10))

### All MiTRI reads

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation
"""

import os
import re
import logging
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
OUTPUT_BASE = "/path/to/project/results/summary_V2/MiTRI_reads_sample_level"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Set[str]:
    """
    Extract reads from MiTRI BAM.txt files(textread_id)
    Returns:
    reads_set: {read_id1, read_id2, ...}
    """
    reads_set = set()
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_set.add(read_id)
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_set


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> tuple:
"""
Aggregate sample MiTRI reads and record microbe sources and taxid information for each read
Returns:
aggregated_reads: {read_id1, read_id2, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
aggregated_reads = set()
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id in microbe_reads:
            aggregated_reads.add(read_id)
            read_to_microbes[read_id].append(microbe)
            # textmicrobealltaxids
            for taxid in microbe_taxids:
                read_to_taxids[read_id].add(taxid)
                # texttaxidsforlisttextaftertextprocess
                read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                return aggregated_reads, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads(cancer: str, status: str, sample: str,
    aggregated_reads: Set[str],
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> List[Dict]:
"""
savesampleMiTRI readsdetailedinformation
Returns:
records: allreadsrecordlist
"""
# createsamplefileoutputpath
sample_file = output_dir / cancer / status / f"{sample}_mitri_reads.tsv"
sample_file.parent.mkdir(parents=True, exist_ok=True)
# buildrecords
records = []
for read_id in aggregated_reads:
    microbes = read_to_microbes.get(read_id, [])
    taxids = read_to_taxids.get(read_id, [])
    records.append({
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'read_id': read_id,
        'microbes_used': '|'.join(microbes),
        'num_microbes': len(microbes),
        'taxids': '|'.join(map(str, taxids)),
        'num_taxids': len(taxids)
    })
    # savetofile
    if records:
        df = pd.DataFrame(records)
        df.to_csv(sample_file, sep='\t', index=False)
        return records


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI reads
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    aggregated_reads, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(aggregated_reads) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None, []
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(aggregated_reads)} from {num_microbes_used} microbes")
    # savesampleMiTRI reads
    records = save_sample_mitri_reads(cancer, status, sample, aggregated_reads,
        read_to_microbes, read_to_taxids, output_dir)
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {len(records)} reads")
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(aggregated_reads),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in aggregated_reads) / len(aggregated_reads),
        'multi_match_reads': sum(1 for r in aggregated_reads if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in aggregated_reads if len(read_to_microbes[r]) > 1) / len(aggregated_reads) * 100
    }
    return summary, records


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    cancer_output = output_base / cancer
    cancer_output.mkdir(parents=True, exist_ok=True)
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            all_records = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary, records in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        all_records.extend(records)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = cancer_output / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            # savecancer typetextallreadstext
                            if all_records:
                                df_all = pd.DataFrame(all_records)
                                all_reads_file = cancer_output / f"{cancer}_all_samples_mitri_reads.tsv"
                                df_all.to_csv(all_reads_file, sep='\t', index=False)
                                logger.info(f"Saved {len(all_records)} reads for {cancer}")
                                logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                                return all_summaries, all_records


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, df_all_reads: pd.DataFrame,
    output_dir: Path):
"""generatedetailedstatistics report"""
report_file = output_dir / "mitri_reads_statistics_report.txt"
with open(report_file, 'w') as f:
    f.write("="*80 + "\n")
    f.write("MiTRI Reads sampletextstatistics report\n")
    f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*80 + "\n\n")
    # Overall statistics
    f.write("[1]Overall statistics\n")
    f.write("-"*80 + "\n")
    f.write(f"textsample count: {len(df_summary)}\n")
    f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
    f.write(f"\ntextMiTRI readsrecordtext: {len(df_all_reads)}\n")
    f.write(f"uniqueread_idtext: {df_all_reads['read_id'].nunique()}\n")
    f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
    f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
    f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
    f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
    f.write("\n")
    # textmatcheddetailedstatistics
    f.write("[2]textmatchedreadsstatistics\n")
    f.write("-"*80 + "\n")
    multi_match = df_all_reads[df_all_reads['num_microbes'] > 1]
    f.write(f"textmatchedreadsrecordtext: {len(multi_match)} ({len(multi_match)/len(df_all_reads)*100:.2f}%)\n")
    f.write(f"text {df_all_reads['num_microbes'].max()} microbestext\n")
    f.write(f"\ntextmatchedtext:\n")
    for i in range(2, min(11, df_all_reads['num_microbes'].max() + 1)):
        count = len(df_all_reads[df_all_reads['num_microbes'] == i])
        if count > 0:
            f.write(f" text{i}microbestext: {count} records ({count/len(df_all_reads)*100:.2f}%)\n")
            f.write("\n")
            # Summarize by cancer type
            f.write("[3]statistics by cancer type\n")
            f.write("-"*80 + "\n")
            cancer_stats = df_summary.groupby('cancer').agg({
                'sample': 'count',
                'total_mitri_reads': ['mean', 'median'],
                'num_microbes': 'mean',
                'multi_match_pct': 'mean'
            }).round(2)
            cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
            f.write(cancer_stats.to_string())
            f.write("\n\n")
            # textstatistics
            f.write("[4]Normal vs Tumor statistics\n")
            f.write("-"*80 + "\n")
            status_stats = df_summary.groupby('status').agg({
                'sample': 'count',
                'total_mitri_reads': ['mean', 'median'],
                'num_microbes': 'mean',
                'multi_match_pct': 'mean'
            }).round(2)
            f.write(status_stats.to_string())
            f.write("\n\n")
            # TOPsample
            f.write("[5]Top 10 samples by MiTRI reads\n")
            f.write("-"*80 + "\n")
            top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
                ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
            ]
            f.write(top_samples.to_string(index=False))
            f.write("\n\n")
            f.write("="*80 + "\n")
            f.write("End of report\n")
            f.write("="*80 + "\n")
            print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatistics")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    all_records = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries, cancer_records = process_cancer(
            cancer, output_base, microbe_to_taxids
)
all_summaries.extend(cancer_summaries)
all_records.extend(cancer_records)
# savetext
if all_summaries:
    df_summary = pd.DataFrame(all_summaries)
    summary_file = output_base / "all_samples_summary.tsv"
    df_summary.to_csv(summary_file, sep='\t', index=False)
    print(f"\nOK Saved global summary: {summary_file}")
    # savetextallreads
    if all_records:
        df_all_reads = pd.DataFrame(all_records)
        all_reads_file = output_base / "all_cancers_all_samples_mitri_reads.tsv"
        df_all_reads.to_csv(all_reads_file, sep='\t', index=False)
        print(f"OK Saved merged reads table: {all_reads_file}")
        print(f" Total reads records: {len(df_all_reads)}")
        print(f" Unique read_ids: {df_all_reads['read_id'].nunique()}")
        print(f" Unique samples: {df_all_reads['sample'].nunique()}")
        print(f" Unique cancers: {df_all_reads['cancer'].nunique()}")
        # generatestatistics report
        if all_summaries and all_records:
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, df_all_reads, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\ntextMiTRI readsrecordtext: {len(df_all_reads)}")
            print(f"uniqueread_idtext: {df_all_reads['read_id'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

### MiTRI_reads.fa

In [ ]:
#!/usr/bin/env python3
"""
Sample-level MiTRI read statistics with FASTA output
statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation, outputforfa.gzformat
"""

import os
import re
import logging
import gzip
from pathlib import Path
from collections import defaultdict
from typing import Dict, Set, List, Tuple
import pandas as pd
from multiprocessing import Pool
import time
from datetime import datetime

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]
N_PROCESSES = 75

# Path configuration
BASE_MITRI = "/path/to/project/results/cancers_V1_V2.8"
BASE_CSV = "/path/to/project/data/CSVs_20251203"
OUTPUT_BASE = "/path/to/project/results/summary_V2/MiTRI_reads_sample_level_fa"
TAXID_HIERARCHY_FILE = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"


def load_taxid_mapping(hierarchy_file: str) -> Dict[str, List[int]]:
    """
    Load microbe-name to tax_id mapping from V7_taxid_hierarchy.txt
    Returns:
    {microbe_name: [tax_id1, tax_id2, ...], ...}
    """
    microbe_to_taxids = defaultdict(list)
    try:
        df = pd.read_csv(hierarchy_file, sep='\t')
        # textrows, textspecies_scientific_nametotax_idmapping
        for _, row in df.iterrows():
            species_name = str(row['species_scientific_name'])
            tax_id = int(row['tax_id'])
            # textspeciestextintextandtextfortext
            microbe_name = re.sub(r'[\s/]+', '_', species_name)
            microbe_to_taxids[microbe_name].append(tax_id)
            print(f"OK Loaded taxid mapping: {len(microbe_to_taxids)} unique microbe names")
    except Exception as e:
        print(f"Failed Error loading taxid hierarchy: {e}")
        return dict(microbe_to_taxids)


def setup_logger(cancer: str, output_dir: Path) -> logging.Logger:
    """foreachcancer typetextlogger"""
    logger = logging.getLogger(f"{cancer}_mitri_stats")
    logger.setLevel(logging.INFO)
    if logger.handlers:
        logger.handlers.clear()
        fh = logging.FileHandler(output_dir / f"{cancer}_mitri_stats_processing.log")
        fh.setLevel(logging.INFO)
        formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
        fh.setFormatter(formatter)
        logger.addHandler(fh)
        return logger


def extract_mitri_reads(txt_path: str) -> Dict[str, Dict[str, str]]:
    """
    Extract reads from MiTRI BAM.txt files(read_id, flag, sequence)
    Returns:
    reads_dict: {read_id: {'flag': flag, 'seq': sequence}, ...}
    """
    reads_dict = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                flag = fields[1]
                sequence = fields[9]
                # Treat reads as valid when the sequence is not *
                if sequence != '*':
                    reads_dict[read_id] = {
                        'flag': flag,
                        'seq': sequence
                    }
    except FileNotFoundError:
        pass
    except Exception as e:
        logging.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return reads_dict


def aggregate_mitri_reads_with_info(cancer: str, status: str, sample: str,     microbes: List[str],     microbe_to_taxids: Dict[str, List[int]],
    logger) -> Tuple[Dict, Dict, Dict]:
"""
Aggregate sample MiTRI reads and record microbe source, taxid, flag, and sequence for each read
Returns:
read_info: {read_id: {'flag': flag, 'seq': seq}, ...}
read_to_microbes: {read_id: [microbe1, microbe2, ...], ...}
read_to_taxids: {read_id: [taxid1, taxid2, ...], ...}
"""
read_info = {} # textreadflagandsequence
read_to_microbes = defaultdict(list) # recordeachreadtextmicrobetext
read_to_taxids = defaultdict(set) # recordeachreadtextalltaxids
for microbe in microbes:
    mitri_txt = (Path(BASE_MITRI) / cancer / "03.coverage" / status /         microbe / f"{sample}_rna_output.pathseq_sorted.bam.txt")
    if not mitri_txt.exists():
        continue
    # extracttextmicrobeMiTRI reads
    microbe_reads = extract_mitri_reads(str(mitri_txt))
    if len(microbe_reads) > 0:
        # textmicrobetaxids
        microbe_taxids = microbe_to_taxids.get(microbe, [])
        for read_id, read_data in microbe_reads.items():
            # textreadtextinformation(onlytext, textto)
            if read_id not in read_info:
                read_info[read_id] = read_data
                # recordmicrobe
                read_to_microbes[read_id].append(microbe)
                # textmicrobealltaxids
                for taxid in microbe_taxids:
                    read_to_taxids[read_id].add(taxid)
                    # texttaxidsforlisttextaftertextprocess
                    read_to_taxids = {k: list(v) for k, v in read_to_taxids.items()}
                    return read_info, dict(read_to_microbes), read_to_taxids


def save_sample_mitri_reads_as_fasta(cancer: str, status: str, sample: str,
    read_info: Dict,
    read_to_microbes: Dict,
    read_to_taxids: Dict,
    output_dir: Path) -> int:
"""
savesampleMiTRI readsforFASTAformat(text)
Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
Returns:
savereadscount
"""
# createOutput filespath
output_dir.mkdir(parents=True, exist_ok=True)
fasta_file = output_dir / f"{cancer}.{status}.{sample}.MiTRI_reads.fa.gz"
if not read_info:
    return 0
    count = 0
    try:
        with gzip.open(fasta_file, 'wt') as f:
            for read_id in sorted(read_info.keys()):
                # textreadinformation
                flag = read_info[read_id]['flag']
                sequence = read_info[read_id]['seq']
                # textmicrobeandtaxids
                microbes = read_to_microbes.get(read_id, [])
                taxids = read_to_taxids.get(read_id, [])
                # buildannotationrows
                # microbestext
                microbes_str = ','.join(microbes)
                # taxidsformat: YP:Z:taxid1,taxid2,...
                taxids_str = f"YP:Z:{','.join(map(str, taxids))}" if taxids else "YP:Z:"
                # textFASTAannotationrows
                # >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids
                header = f">{cancer}|{status}|{sample}|{read_id}|{flag}|{microbes_str}|{len(microbes)}|{len(taxids)}|{taxids_str}"
                # textFASTAformat
                f.write(f"{header}\n")
                f.write(f"{sequence}\n")
                count += 1
    except Exception as e:
        logging.error(f"Error writing FASTA file {fasta_file}: {e}")
        return 0
    return count


def process_sample(args):
    """
    processtextsamples: statisticsallMiTRI readstextsaveforFASTAformat
    """
    cancer, status, sample, microbes, microbe_to_taxids, output_dir, logger_name = args
    logger = logging.getLogger(logger_name)
    logger.info(f"Processing {cancer}/{status}/{sample}")
    # textMiTRI reads(textdetailedinformation)
    read_info, read_to_microbes, read_to_taxids = aggregate_mitri_reads_with_info(
        cancer, status, sample, microbes, microbe_to_taxids, logger
)
if len(read_info) == 0:
    logger.warning(f"{sample}: No MiTRI reads found")
    return None
    num_microbes_used = len(set([m for microbes_list in read_to_microbes.values()         for m in microbes_list]))
    logger.info(f"{sample}: Total MiTRI reads = {len(read_info)} from {num_microbes_used} microbes")
    # savesampleMiTRI readsforFASTAformat
    saved_count = save_sample_mitri_reads_as_fasta(
        cancer, status, sample, read_info,
        read_to_microbes, read_to_taxids, output_dir
)
if saved_count > 0:
    logger.info(f"Completed {cancer}/{status}/{sample}: saved {saved_count} reads to FASTA")
else:
    logger.error(f"Failed to save FASTA for {cancer}/{status}/{sample}")
    return None
    # returntextstatistics
    summary = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'total_mitri_reads': len(read_info),
        'num_microbes': num_microbes_used,
        'avg_microbes_per_read': sum(len(read_to_microbes[r]) for r in read_info) / len(read_info),
        'multi_match_reads': sum(1 for r in read_info if len(read_to_microbes[r]) > 1),
        'multi_match_pct': sum(1 for r in read_info if len(read_to_microbes[r]) > 1) / len(read_info) * 100
    }
    return summary


def get_sample_list(cancer: str, status: str) -> List[str]:
    """textsample columntext(fromMiTRIresultsdirectory)"""
    mitri_dir = Path(BASE_MITRI) / cancer / "03.coverage" / status
    if not mitri_dir.exists():
        return []
    # fromtextmicrobesdirectorytextsample columntext
    samples = set()
    for microbe_dir in mitri_dir.iterdir():
        if microbe_dir.is_dir():
            for txt_file in microbe_dir.glob("*_rna_output.pathseq_sorted.bam.txt"):
                sample = txt_file.name.replace("_rna_output.pathseq_sorted.bam.txt", "")
                samples.add(sample)
                return sorted(list(samples))


def get_microbe_list(cancer: str) -> List[str]:
    """fromCSVfiletextmicrobe list"""
    csv_file = Path(BASE_CSV) / f"{cancer}_V3.csv"
    if not csv_file.exists():
        return []
    df = pd.read_csv(csv_file)
    return df['Taxon'].tolist()


def process_cancer(cancer: str, output_base: Path, microbe_to_taxids: Dict):
    """processtextcancer typeallsample"""
    logger = setup_logger(cancer, output_base)
    logger.info(f"=== Starting processing for {cancer} ===")
    microbes = get_microbe_list(cancer)
    logger.info(f"Found {len(microbes)} microbes for {cancer}")
    # textalltext
    tasks = []
    for status in STATUSES:
        samples = get_sample_list(cancer, status)
        logger.info(f"{cancer}/{status}: {len(samples)} samples")
        for sample in samples:
            tasks.append((cancer, status, sample, microbes, microbe_to_taxids,                 output_base, f"{cancer}_mitri_stats"))
            logger.info(f"Total tasks for {cancer}: {len(tasks)}")
            # textrowsprocess
            all_summaries = []
            with Pool(processes=min(N_PROCESSES, len(tasks))) as pool:
                for summary in pool.imap_unordered(process_sample, tasks):
                    if summary is not None:
                        all_summaries.append(summary)
                        # savecancer typetextstatistics
                        if all_summaries:
                            df_summary = pd.DataFrame(all_summaries)
                            summary_file = output_base / f"{cancer}_summary.tsv"
                            df_summary.to_csv(summary_file, sep='\t', index=False)
                            logger.info(f"Saved cancer summary: {summary_file}")
                            logger.info(f"=== Completed {cancer}: {len(all_summaries)} samples ===")
                            return all_summaries


def create_visualizations(df_summary: pd.DataFrame, output_dir: Path):
    """createstatisticstext"""
    try:
        import matplotlib.pyplot as plt
        import seaborn as sns
        sns.set_style("whitegrid")
        fig, axes = plt.subplots(2, 3, figsize=(20, 12))
        # 1. Summarize by cancer typeMiTRI readscount
        ax1 = axes[0, 0]
        cancer_stats = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
        cancer_stats.plot(kind='barh', ax=ax1, color='steelblue')
        ax1.set_xlabel('Average MiTRI Reads Count')
        ax1.set_title('Average MiTRI Reads by Cancer Type')
        ax1.grid(axis='x', alpha=0.3)
        # 2. MiTRI readscounttext
        ax2 = axes[0, 1]
        df_summary['total_mitri_reads'].hist(bins=50, ax=ax2, color='green', edgecolor='black')
        ax2.set_xlabel('MiTRI Reads Count')
        ax2.set_ylabel('Frequency')
        ax2.set_title('Distribution of MiTRI Reads Count')
        ax2.axvline(df_summary['total_mitri_reads'].median(), color='red', linestyle='--',
            label=f"Median: {df_summary['total_mitri_reads'].median():.0f}")
        ax2.legend()
        # 3. Normal vs Tumorcompare
        ax3 = axes[0, 2]
        status_stats = df_summary.groupby('status')['total_mitri_reads'].mean()
        status_stats.plot(kind='bar', ax=ax3, color=['lightblue', 'coral'])
        ax3.set_xlabel('Status')
        ax3.set_ylabel('Average MiTRI Reads')
        ax3.set_title('MiTRI Reads: Normal vs Tumor')
        ax3.set_xticklabels(ax3.get_xticklabels(), rotation=0)
        ax3.grid(axis='y', alpha=0.3)
        # 4. microbecounttext
        ax4 = axes[1, 0]
        df_summary['num_microbes'].hist(bins=30, ax=ax4, color='purple', edgecolor='black')
        ax4.set_xlabel('Number of Microbes')
        ax4.set_ylabel('Frequency')
        ax4.set_title('Distribution of Microbes Count per Sample')
        ax4.axvline(df_summary['num_microbes'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['num_microbes'].mean():.1f}")
        ax4.legend()
        # 5. textmatchedreadstext
        ax5 = axes[1, 1]
        df_summary['multi_match_pct'].hist(bins=50, ax=ax5, color='orange', edgecolor='black')
        ax5.set_xlabel('Multi-Match Reads %')
        ax5.set_ylabel('Frequency')
        ax5.set_title('Distribution of Multi-Match Reads %')
        ax5.axvline(df_summary['multi_match_pct'].mean(), color='red', linestyle='--',
            label=f"Mean: {df_summary['multi_match_pct'].mean():.1f}%")
        ax5.legend()
        # 6. microbecount vs MiTRI reads
        ax6 = axes[1, 2]
        scatter = ax6.scatter(df_summary['num_microbes'], df_summary['total_mitri_reads'],
            c=df_summary['multi_match_pct'], cmap='viridis', alpha=0.6, s=30)
        ax6.set_xlabel('Number of Microbes')
        ax6.set_ylabel('Total MiTRI Reads')
        ax6.set_title('Microbes Count vs MiTRI Reads')
        plt.colorbar(scatter, ax=ax6, label='Multi-Match %')
        ax6.grid(alpha=0.3)
        plt.tight_layout()
        plt.savefig(output_dir / 'mitri_reads_visualization.png', dpi=300, bbox_inches='tight')
        plt.close()
        print("OK Visualization saved successfully")
    except ImportError:
        print("Warning Matplotlib/Seaborn not available, skipping visualization")
    except Exception as e:
        print(f"Failed Error creating visualization: {e}")


def generate_statistics_report(df_summary: pd.DataFrame, output_dir: Path):
    """generatedetailedstatistics report"""
    report_file = output_dir / "mitri_reads_statistics_report.txt"
    with open(report_file, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MiTRI Reads sampletextstatistics report (FASTAformatoutput)\n")
        f.write(f"Generated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        f.write("="*80 + "\n\n")
        # Overall statistics
        f.write("[1]Overall statistics\n")
        f.write("-"*80 + "\n")
        f.write(f"textsample count: {len(df_summary)}\n")
        f.write(f"total cancer types: {df_summary['cancer'].nunique()}\n")
        f.write(f"\naverage MiTRI reads per sample: {df_summary['total_mitri_reads'].mean():.0f}\n")
        f.write(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}\n")
        f.write(f"average microbes using each read: {df_summary['avg_microbes_per_read'].mean():.2f}\n")
        f.write(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%\n")
        f.write("\n")
        # Summarize by cancer type
        f.write("[2]statistics by cancer type\n")
        f.write("-"*80 + "\n")
        cancer_stats = df_summary.groupby('cancer').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        cancer_stats.columns = ['sample count', 'average reads', 'median reads', 'average microbes', 'averagetextmatched%']
        f.write(cancer_stats.to_string())
        f.write("\n\n")
        # textstatistics
        f.write("[3]Normal vs Tumor statistics\n")
        f.write("-"*80 + "\n")
        status_stats = df_summary.groupby('status').agg({
            'sample': 'count',
            'total_mitri_reads': ['mean', 'median'],
            'num_microbes': 'mean',
            'multi_match_pct': 'mean'
        }).round(2)
        f.write(status_stats.to_string())
        f.write("\n\n")
        # TOPsample
        f.write("[4]Top 10 samples by MiTRI reads\n")
        f.write("-"*80 + "\n")
        top_samples = df_summary.nlargest(10, 'total_mitri_reads')[
            ['cancer', 'status', 'sample', 'total_mitri_reads', 'num_microbes']
        ]
        f.write(top_samples.to_string(index=False))
        f.write("\n\n")
        # outputformattext
        f.write("[5]Output file format notes\n")
        f.write("-"*80 + "\n")
        f.write("File format: FASTA (gzip compressed)\n")
        f.write("Filename: {cancer}.{status}.{sample}.MiTRI_reads.fa.gz\n")
        f.write("Annotation format: >cancer|status|sample|read_id|flag|microbes_used|num_microbes|num_taxids|taxids\n")
        f.write(" - microbes_used: textmicrobe list\n")
        f.write(" - taxids: YP:Z:prefix followed by a comma-separated taxid list\n")
        f.write("\ntext:\n")
        f.write(">BRCA|Normal|SRR11600335|SRR11600335.35235387|16|Acinetobacter_johnsonii,Enterobacter_asburiae|2|2|YP:Z:61645,1217662\n")
        f.write("ATCGATCGATCG...\n")
        f.write("\n")
        f.write("="*80 + "\n")
        f.write("End of report\n")
        f.write("="*80 + "\n")
        print(f"OK Statistics report saved: {report_file}")


def main():
    """text"""
    start_time = time.time()
    print(f"\n{'='*80}")
    print(f"MiTRI Reads sampletextstatisticstextoutputforFASTAformat")
    print(f"statisticstextsamplesallMiTRI reads, recordmicrobeandtaxidinformation")
    print(f"starttext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"{'='*80}\n")
    # loadtaxidmapping
    print("textloadmicrobetaxidmapping...")
    microbe_to_taxids = load_taxid_mapping(TAXID_HIERARCHY_FILE)
    # Create output directory
    output_base = Path(OUTPUT_BASE)
    output_base.mkdir(parents=True, exist_ok=True)
    # processeachcancer type
    all_summaries = []
    for cancer in CANCERS:
        print(f"\n{'='*80}")
        print(f"Processing {cancer}...")
        print(f"{'='*80}")
        cancer_summaries = process_cancer(cancer, output_base, microbe_to_taxids)
        all_summaries.extend(cancer_summaries)
        # savetext
        if all_summaries:
            df_summary = pd.DataFrame(all_summaries)
            summary_file = output_base / "all_samples_summary.tsv"
            df_summary.to_csv(summary_file, sep='\t', index=False)
            print(f"\nOK Saved global summary: {summary_file}")
            # generatestatistics report
            print("\n" + "="*80)
            print("generatestatistics report...")
            print("="*80)
            generate_statistics_report(df_summary, output_base)
            # textoutputtextstatistics
            print("\n" + "="*80)
            print("textstatistics")
            print("="*80)
            print(f"textsample count: {len(df_summary)}")
            print(f"total cancer types: {df_summary['cancer'].nunique()}")
            print(f"\naveragetextsampleMiTRI reads: {df_summary['total_mitri_reads'].mean():.0f}")
            print(f"average microbes per sample: {df_summary['num_microbes'].mean():.1f}")
            print(f"averagetextmatchedreadstext: {df_summary['multi_match_pct'].mean():.2f}%")
            print(f"\n[textcancer typeaverageMiTRI reads]")
            cancer_avg = df_summary.groupby('cancer')['total_mitri_reads'].mean().sort_values()
            for cancer, avg in cancer_avg.items():
                print(f" {cancer:6s}: {avg:8.0f}")
                print(f"\n[Normal vs Tumor]")
                status_avg = df_summary.groupby('status')['total_mitri_reads'].mean()
                for status, avg in status_avg.items():
                    print(f" {status:6s}: {avg:8.0f}")
                    # createtext
                    print("\n" + "="*80)
                    print("generatetext...")
                    print("="*80)
                    create_visualizations(df_summary, output_base)
                    elapsed_time = time.time() - start_time
                    print(f"\n{'='*80}")
                    print(f"MiTRI read statisticscompleted!")
                    print(f"output directory: {OUTPUT_BASE}")
                    print(f"text: {elapsed_time/60:.2f} text")
                    print(f"endtext: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
                    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()

# code V3

In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
BAM read intersection statistics analysis (enhanced version)
comparePathSeqandMiTRItextreads, texttaxidtext
Support cross-sample deduplicated statistics and visualization
"""

import os
import re
import pysam
import pandas as pd
from collections import defaultdict
from multiprocessing import Pool, cpu_count
import json
from pathlib import Path
import logging
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Set plotting font and style
plt.rcParams['font.sans-serif'] = ['Arial', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# Global configuration
CANCERS = ["LUNG", "OSCC", "THCA", "CRC", "BRCA", "CESC", "LIHC", "STAD", "PAAD"]
STATUSES = ["Normal", "Tumor"]

# Path configuration
PATHSEQ_BASE = "/path/to/project/data/cancers_V1"
MITRI_BASE = "/path/to/project/results/cancers_V1_V2.8"
MICROBE_CSV_DIR = "/path/to/project/data/CSVs_20251203"
TAXID_HIERARCHY = "/path/to/project/data/genome_annotation/V7_taxid_hierarchy.txt"
OUTPUT_DIR = "/path/to/project/results/summary_V2/reads_compare"

# statistics mode: 'cumulative'(text)or 'deduplicated'(deduplicated across samples)
STAT_MODE = 'cumulative' # can be changed to 'deduplicated'


def normalize_microbe_name(name):
    """textmicrobetext"""
    return re.sub(r'[\s/\\]+', '_', name)


def load_taxid_hierarchy():
    """loadtaxidtext"""
    logger.info("Loading taxid hierarchy...")
    df = pd.read_csv(TAXID_HIERARCHY, sep='\t')
    microbe_to_taxids = defaultdict(list)
    for _, row in df.iterrows():
        species_name = row['species_scientific_name']
        if pd.notna(species_name):
            normalized_name = normalize_microbe_name(species_name)
            tax_id = str(int(row['tax_id']))
            microbe_to_taxids[normalized_name].append(tax_id)
            logger.info(f"Loaded {len(microbe_to_taxids)} microbe-taxid mappings")
            return dict(microbe_to_taxids)


def load_microbe_list(cancer):
    """loadtextcancer typemicrobe list"""
    csv_path = os.path.join(MICROBE_CSV_DIR, f"{cancer}_V3.csv")
    if not os.path.exists(csv_path):
        logger.warning(f"Microbe list not found: {csv_path}")
        return []
    df = pd.read_csv(csv_path)
    microbes = df['Taxon'].tolist()
    logger.info(f"{cancer}: Found {len(microbes)} microbes")
    return microbes


def extract_pathseq_reads(bam_path):
    """Extract reads with YP tags from a PathSeq BAM file"""
    if not os.path.exists(bam_path):
        logger.warning(f"PathSeq BAM not found: {bam_path}")
        return {}
    pathseq_reads = {}
    try:
        with pysam.AlignmentFile(bam_path, "rb", check_sq=False) as bam:
            for read in bam:
                if read.has_tag('YP'):
                    yp_value = read.get_tag('YP')
                    taxids = [tid.strip() for tid in str(yp_value).split(',')]
                    sequence = read.query_sequence
                    if sequence is None:
                        continue
                    read_id = read.query_name
                    key = (read_id, sequence)
                    pathseq_reads[key] = taxids
                    logger.info(f" PathSeq: {len(pathseq_reads)} reads with YP tag")
    except Exception as e:
        logger.error(f"Error reading PathSeq BAM {bam_path}: {e}")
        return pathseq_reads


def extract_mitri_reads(txt_path):
    """Extract reads from MiTRI .bam.txt files"""
    if not os.path.exists(txt_path):
        return {}
    mitri_reads = {}
    try:
        with open(txt_path, 'r') as f:
            for line in f:
                if line.startswith('@'):
                    continue
                fields = line.strip().split('\t')
                if len(fields) < 10:
                    continue
                read_id = fields[0]
                sequence = fields[9]
                key = (read_id, sequence)
                mitri_reads[key] = True
    except Exception as e:
        logger.error(f"Error reading MiTRI txt {txt_path}: {e}")
        return mitri_reads


def process_sample(args):
    """processtextsamples"""
    cancer, status, sample, microbe_list, microbe_to_taxids = args
    logger.info(f"Processing {cancer}-{status}-{sample}")
    pathseq_bam = os.path.join(
        PATHSEQ_BASE, cancer, status, "bam",
        f"{sample}_rna_output.pathseq.bam"
)
pathseq_reads = extract_pathseq_reads(pathseq_bam)
if not pathseq_reads:
    return None
    sample_stats = {
        'cancer': cancer,
        'status': status,
        'sample': sample,
        'pathseq_total': len(pathseq_reads),
        'microbes': {},
        'intersection_reads': {} # textdeduplicated across samples
    }
    for microbe in microbe_list:
        mitri_txt = os.path.join(
            MITRI_BASE, cancer, "03.coverage", status, microbe,
            f"{sample}_rna_output.pathseq_sorted.bam.txt"
)
mitri_reads = extract_mitri_reads(mitri_txt)
if not mitri_reads:
    continue
    intersection = set(pathseq_reads.keys()) & set(mitri_reads.keys())
    if not intersection:
        continue
    microbe_taxids = set(microbe_to_taxids.get(microbe, []))
    matched_reads = set()
    unmatched_reads = set()
    for key in intersection:
        pathseq_taxids = set(pathseq_reads[key])
        if pathseq_taxids & microbe_taxids:
            matched_reads.add(key)
        else:
            unmatched_reads.add(key)
            sample_stats['microbes'][microbe] = {
                'mitri_total': len(mitri_reads),
                'intersection_total': len(intersection),
                'taxid_matched': len(matched_reads),
                'taxid_unmatched': len(unmatched_reads)
            }
            # savereadstextdeduplicated across samples
            sample_stats['intersection_reads'][microbe] = {
                'all': intersection,
                'matched': matched_reads,
                'unmatched': unmatched_reads
            }
            return sample_stats


def get_all_samples(cancer, status):
    """textallsample"""
    bam_dir = os.path.join(PATHSEQ_BASE, cancer, status, "bam")
    if not os.path.exists(bam_dir):
        return []
    samples = []
    for f in os.listdir(bam_dir):
        if f.endswith('_rna_output.pathseq.bam'):
            sample = f.replace('_rna_output.pathseq.bam', '')
            samples.append(sample)
            return samples


def summarize_results_cumulative(all_results):
    """text(textfirstsampletextdeduplicate, textsampletext)"""
    cancer_summary = defaultdict(lambda: {
        'total_samples': 0,
        'total_pathseq_reads': 0,
        'microbes': defaultdict(lambda: {
        'intersection_total': 0,
        'taxid_matched': 0,
        'taxid_unmatched': 0,
        'match_percentage': 0.0
    })
    })
    global_summary = {
        'total_samples': 0,
        'total_pathseq_reads': 0,
        'microbes': defaultdict(lambda: {
        'intersection_total': 0,
        'taxid_matched': 0,
        'taxid_unmatched': 0,
        'match_percentage': 0.0
    })
    }
    for result in all_results:
        if result is None:
            continue
        cancer = result['cancer']
        cancer_summary[cancer]['total_samples'] += 1
        cancer_summary[cancer]['total_pathseq_reads'] += result['pathseq_total']
        for microbe, stats in result['microbes'].items():
            cancer_summary[cancer]['microbes'][microbe]['intersection_total'] += stats['intersection_total']
            cancer_summary[cancer]['microbes'][microbe]['taxid_matched'] += stats['taxid_matched']
            cancer_summary[cancer]['microbes'][microbe]['taxid_unmatched'] += stats['taxid_unmatched']
            global_summary['total_samples'] += 1
            global_summary['total_pathseq_reads'] += result['pathseq_total']
            for microbe, stats in result['microbes'].items():
                global_summary['microbes'][microbe]['intersection_total'] += stats['intersection_total']
                global_summary['microbes'][microbe]['taxid_matched'] += stats['taxid_matched']
                global_summary['microbes'][microbe]['taxid_unmatched'] += stats['taxid_unmatched']
                # calculatetext
                for cancer in cancer_summary:
                    for microbe in cancer_summary[cancer]['microbes']:
                        total = cancer_summary[cancer]['microbes'][microbe]['intersection_total']
                        matched = cancer_summary[cancer]['microbes'][microbe]['taxid_matched']
                        if total > 0:
                            cancer_summary[cancer]['microbes'][microbe]['match_percentage'] = (matched / total) * 100
                            for microbe in global_summary['microbes']:
                                total = global_summary['microbes'][microbe]['intersection_total']
                                matched = global_summary['microbes'][microbe]['taxid_matched']
                                if total > 0:
                                    global_summary['microbes'][microbe]['match_percentage'] = (matched / total) * 100
                                    return cancer_summary, global_summary


def summarize_results_deduplicated(all_results):
    """deduplicatetext(deduplicated across samplesunique reads)"""
    cancer_summary = defaultdict(lambda: {
        'total_samples': 0,
        'total_pathseq_reads': 0,
        'microbes': defaultdict(lambda: {
        'intersection_reads': set(),
        'matched_reads': set(),
        'unmatched_reads': set()
    })
    })
    global_summary = {
        'total_samples': 0,
        'total_pathseq_reads': 0,
        'microbes': defaultdict(lambda: {
        'intersection_reads': set(),
        'matched_reads': set(),
        'unmatched_reads': set()
    })
    }
    for result in all_results:
        if result is None:
            continue
        cancer = result['cancer']
        cancer_summary[cancer]['total_samples'] += 1
        cancer_summary[cancer]['total_pathseq_reads'] += result['pathseq_total']
        for microbe, reads_dict in result['intersection_reads'].items():
            cancer_summary[cancer]['microbes'][microbe]['intersection_reads'].update(reads_dict['all'])
            cancer_summary[cancer]['microbes'][microbe]['matched_reads'].update(reads_dict['matched'])
            cancer_summary[cancer]['microbes'][microbe]['unmatched_reads'].update(reads_dict['unmatched'])
            global_summary['microbes'][microbe]['intersection_reads'].update(reads_dict['all'])
            global_summary['microbes'][microbe]['matched_reads'].update(reads_dict['matched'])
            global_summary['microbes'][microbe]['unmatched_reads'].update(reads_dict['unmatched'])
            global_summary['total_samples'] += 1
            global_summary['total_pathseq_reads'] += result['pathseq_total']
            # textfortext
            for cancer in cancer_summary:
                for microbe in cancer_summary[cancer]['microbes']:
                    total = len(cancer_summary[cancer]['microbes'][microbe]['intersection_reads'])
                    matched = len(cancer_summary[cancer]['microbes'][microbe]['matched_reads'])
                    unmatched = len(cancer_summary[cancer]['microbes'][microbe]['unmatched_reads'])
                    cancer_summary[cancer]['microbes'][microbe] = {
                        'intersection_total': total,
                        'taxid_matched': matched,
                        'taxid_unmatched': unmatched,
                        'match_percentage': (matched / total * 100) if total > 0 else 0.0
                    }
                    for microbe in global_summary['microbes']:
                        total = len(global_summary['microbes'][microbe]['intersection_reads'])
                        matched = len(global_summary['microbes'][microbe]['matched_reads'])
                        unmatched = len(global_summary['microbes'][microbe]['unmatched_reads'])
                        global_summary['microbes'][microbe] = {
                            'intersection_total': total,
                            'taxid_matched': matched,
                            'taxid_unmatched': unmatched,
                            'match_percentage': (matched / total * 100) if total > 0 else 0.0
                        }
                        return cancer_summary, global_summary


def create_visualizations(cancer_summary, global_summary, output_dir):
    """createtext"""
    vis_dir = os.path.join(output_dir, "visualizations")
    os.makedirs(vis_dir, exist_ok=True)
    # 1. textTopmicrobetext
    logger.info("Creating global top microbes bar chart...")
    microbe_data = []
    for microbe, stats in global_summary['microbes'].items():
        microbe_data.append({
            'Microbe': microbe,
            'Intersection': stats['intersection_total'],
            'Matched': stats['taxid_matched'],
            'Unmatched': stats['taxid_unmatched'],
            'Match %': stats['match_percentage']
        })
        df_global = pd.DataFrame(microbe_data)
        df_global = df_global.sort_values('Intersection', ascending=False).head(20)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 8))
        # text: textcount
        x_pos = np.arange(len(df_global))
        ax1.bar(x_pos, df_global['Matched'], label='Taxid Matched', color='#2ecc71', alpha=0.8)
        ax1.bar(x_pos, df_global['Unmatched'], bottom=df_global['Matched'],             label='Taxid Unmatched', color='#e74c3c', alpha=0.8)
        ax1.set_xlabel('Microbe', fontsize=12)
        ax1.set_ylabel('Number of Reads', fontsize=12)
        ax1.set_title('Top 20 Microbes - Intersection Reads Distribution', fontsize=14, fontweight='bold')
        ax1.set_xticks(x_pos)
        ax1.set_xticklabels(df_global['Microbe'], rotation=45, ha='right', fontsize=9)
        ax1.legend()
        ax1.grid(axis='y', alpha=0.3)
        # text: matchedtext
        colors = ['#2ecc71' if x >= 50 else '#f39c12' if x >= 20 else '#e74c3c'             for x in df_global['Match %']]
        ax2.barh(x_pos, df_global['Match %'], color=colors, alpha=0.8)
        ax2.set_ylabel('Microbe', fontsize=12)
        ax2.set_xlabel('Taxid Match Percentage (%)', fontsize=12)
        ax2.set_title('Top 20 Microbes - Taxid Match Rate', fontsize=14, fontweight='bold')
        ax2.set_yticks(x_pos)
        ax2.set_yticklabels(df_global['Microbe'], fontsize=9)
        ax2.set_xlim(0, 100)
        ax2.grid(axis='x', alpha=0.3)
        # text
        for i, v in enumerate(df_global['Match %']):
            ax2.text(v + 2, i, f'{v:.1f}%', va='center', fontsize=8)
            plt.tight_layout()
            plt.savefig(os.path.join(vis_dir, 'global_top20_microbes.png'), dpi=300, bbox_inches='tight')
            plt.close()
            # 2. textcancer typeOverall statisticstext
            logger.info("Creating cancer type comparison...")
            cancer_data = []
            for cancer, stats in cancer_summary.items():
                total_intersection = sum(m['intersection_total'] for m in stats['microbes'].values())
                total_matched = sum(m['taxid_matched'] for m in stats['microbes'].values())
                total_unmatched = sum(m['taxid_unmatched'] for m in stats['microbes'].values())
                match_rate = (total_matched / total_intersection * 100) if total_intersection > 0 else 0
                cancer_data.append({
                    'Cancer': cancer,
                    'Samples': stats['total_samples'],
                    'PathSeq Reads': stats['total_pathseq_reads'],
                    'Intersection': total_intersection,
                    'Matched': total_matched,
                    'Unmatched': total_unmatched,
                    'Match %': match_rate
                })
                df_cancer = pd.DataFrame(cancer_data).sort_values('Intersection', ascending=False)
                fig, axes = plt.subplots(2, 2, figsize=(16, 12))
                # text1: sample count
                axes[0, 0].bar(df_cancer['Cancer'], df_cancer['Samples'], color='#3498db', alpha=0.8)
                axes[0, 0].set_title('Number of Samples per Cancer Type', fontsize=12, fontweight='bold')
                axes[0, 0].set_ylabel('Samples', fontsize=10)
                axes[0, 0].tick_params(axis='x', rotation=45)
                axes[0, 0].grid(axis='y', alpha=0.3)
                # text2: PathSeqtextreads
                axes[0, 1].bar(df_cancer['Cancer'], df_cancer['PathSeq Reads'], color='#9b59b6', alpha=0.8)
                axes[0, 1].set_title('Total PathSeq Reads per Cancer Type', fontsize=12, fontweight='bold')
                axes[0, 1].set_ylabel('Reads', fontsize=10)
                axes[0, 1].tick_params(axis='x', rotation=45)
                axes[0, 1].grid(axis='y', alpha=0.3)
                axes[0, 1].ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
                # text3: text
                x_pos = np.arange(len(df_cancer))
                axes[1, 0].bar(x_pos, df_cancer['Matched'], label='Matched', color='#2ecc71', alpha=0.8)
                axes[1, 0].bar(x_pos, df_cancer['Unmatched'], bottom=df_cancer['Matched'],                     label='Unmatched', color='#e74c3c', alpha=0.8)
                axes[1, 0].set_title('Intersection Reads Distribution', fontsize=12, fontweight='bold')
                axes[1, 0].set_ylabel('Reads', fontsize=10)
                axes[1, 0].set_xticks(x_pos)
                axes[1, 0].set_xticklabels(df_cancer['Cancer'], rotation=45)
                axes[1, 0].legend()
                axes[1, 0].grid(axis='y', alpha=0.3)
                # text4: matchedtext
                colors_cancer = ['#2ecc71' if x >= 50 else '#f39c12' if x >= 20 else '#e74c3c'                     for x in df_cancer['Match %']]
                axes[1, 1].bar(df_cancer['Cancer'], df_cancer['Match %'], color=colors_cancer, alpha=0.8)
                axes[1, 1].set_title('Taxid Match Rate by Cancer Type', fontsize=12, fontweight='bold')
                axes[1, 1].set_ylabel('Match %', fontsize=10)
                axes[1, 1].tick_params(axis='x', rotation=45)
                axes[1, 1].set_ylim(0, 100)
                axes[1, 1].grid(axis='y', alpha=0.3)
                plt.tight_layout()
                plt.savefig(os.path.join(vis_dir, 'cancer_comparison.png'), dpi=300, bbox_inches='tight')
                plt.close()
                # 3. text: textcancer typemicrobematchedtext
                logger.info("Creating heatmap...")
                heatmap_data = []
                all_microbes = set()
                for cancer, stats in cancer_summary.items():
                    all_microbes.update(stats['microbes'].keys())
                    # onlytextcancer typeintexttopmicrobe
                    microbe_cancer_count = defaultdict(int)
                    for cancer, stats in cancer_summary.items():
                        for microbe in stats['microbes'].keys():
                            microbe_cancer_count[microbe] += 1
                            common_microbes = [m for m, c in sorted(microbe_cancer_count.items(),                                 key=lambda x: x[1], reverse=True)][:30]
                            heatmap_matrix = []
                            for microbe in common_microbes:
                                row = []
                                for cancer in CANCERS:
                                    if microbe in cancer_summary[cancer]['microbes']:
                                        match_pct = cancer_summary[cancer]['microbes'][microbe]['match_percentage']
                                        row.append(match_pct)
                                    else:
                                        row.append(np.nan)
                                        heatmap_matrix.append(row)
                                        df_heatmap = pd.DataFrame(heatmap_matrix, index=common_microbes, columns=CANCERS)
                                        plt.figure(figsize=(12, 16))
                                        sns.heatmap(df_heatmap, annot=True, fmt='.1f', cmap='RdYlGn',                                             cbar_kws={'label': 'Match Percentage (%)'},                                             vmin=0, vmax=100, linewidths=0.5)
                                        plt.title('Taxid Match Rate Heatmap (Top 30 Common Microbes)',                                             fontsize=14, fontweight='bold', pad=20)
                                        plt.xlabel('Cancer Type', fontsize=12)
                                        plt.ylabel('Microbe', fontsize=12)
                                        plt.tight_layout()
                                        plt.savefig(os.path.join(vis_dir, 'match_rate_heatmap.png'), dpi=300, bbox_inches='tight')
                                        plt.close()
                                        logger.info(f"Visualizations saved to {vis_dir}")


def save_results(cancer_summary, global_summary, output_dir):
    """savestatisticsresults"""
    os.makedirs(output_dir, exist_ok=True)
    # savetextcancer typedetailedstatistics
    for cancer, stats in cancer_summary.items():
        output_file = os.path.join(output_dir, f"{cancer}_summary.txt")
        with open(output_file, 'w') as f:
            f.write(f"Cancer Type: {cancer}\n")
            f.write(f"Statistical Mode: {STAT_MODE.upper()}\n")
            f.write(f"Total Samples: {stats['total_samples']}\n")
            f.write(f"Total PathSeq Reads: {stats['total_pathseq_reads']:,}\n\n")
            f.write("=" * 100 + "\n")
            f.write(f"{'Microbe':<45} {'Intersection':<15} {'Matched':<15} {'Unmatched':<15} {'Match %':<10}\n")
            f.write("=" * 100 + "\n")
            for microbe, microbe_stats in sorted(stats['microbes'].items(),                 key=lambda x: x[1]['intersection_total'],                 reverse=True):
            f.write(f"{microbe:<45} {microbe_stats['intersection_total']:<15,} "
                f"{microbe_stats['taxid_matched']:<15,} "
                f"{microbe_stats['taxid_unmatched']:<15,} "
                f"{microbe_stats['match_percentage']:<10.2f}\n")
            logger.info(f"Saved {cancer} summary to {output_file}")
            # savetextstatistics
            global_file = os.path.join(output_dir, "global_summary.txt")
            with open(global_file, 'w') as f:
                f.write("=" * 100 + "\n")
                f.write("GLOBAL SUMMARY ACROSS ALL CANCERS\n")
                f.write("=" * 100 + "\n\n")
                f.write(f"Statistical Mode: {STAT_MODE.upper()}\n")
                if STAT_MODE == 'cumulative':
                    f.write(" (Reads are deduplicated within each sample, then summed across samples)\n\n")
                else:
                    f.write(" (Reads are deduplicated across all samples - unique reads only)\n\n")
                    f.write(f"Total Samples: {global_summary['total_samples']}\n")
                    f.write(f"Total PathSeq Reads: {global_summary['total_pathseq_reads']:,}\n\n")
                    f.write("=" * 100 + "\n")
                    f.write(f"{'Microbe':<45} {'Intersection':<15} {'Matched':<15} {'Unmatched':<15} {'Match %':<10}\n")
                    f.write("=" * 100 + "\n")
                    for microbe, microbe_stats in sorted(global_summary['microbes'].items(),                         key=lambda x: x[1]['intersection_total'],                         reverse=True):
                    f.write(f"{microbe:<45} {microbe_stats['intersection_total']:<15,} "
                        f"{microbe_stats['taxid_matched']:<15,} "
                        f"{microbe_stats['taxid_unmatched']:<15,} "
                        f"{microbe_stats['match_percentage']:<10.2f}\n")
                    logger.info(f"Saved global summary to {global_file}")
                    # savedetailedCSV
                    csv_file = os.path.join(output_dir, "detailed_summary.csv")
                    csv_data = []
                    for cancer, stats in cancer_summary.items():
                        for microbe, microbe_stats in stats['microbes'].items():
                            csv_data.append({
                                'Cancer': cancer,
                                'Microbe': microbe,
                                'Intersection_Total': microbe_stats['intersection_total'],
                                'Taxid_Matched': microbe_stats['taxid_matched'],
                                'Taxid_Unmatched': microbe_stats['taxid_unmatched'],
                                'Match_Percentage': microbe_stats['match_percentage']
                            })
                            df_csv = pd.DataFrame(csv_data)
                            df_csv.to_csv(csv_file, index=False)
                            logger.info(f"Saved detailed CSV to {csv_file}")
                            # saveJSON
                            json_file = os.path.join(output_dir, "summary.json")
                            with open(json_file, 'w') as f:
                                json.dump({
                                    'stat_mode': STAT_MODE,
                                    'cancer_summary': {k: dict(v) for k, v in cancer_summary.items()},
                                    'global_summary': dict(global_summary)
                                }, f, indent=2)
                                logger.info(f"Saved JSON summary to {json_file}")


def main():
    """text"""
    logger.info("=" * 80)
    logger.info("Starting BAM Intersection Analysis (Enhanced Version)")
    logger.info(f"Statistical Mode: {STAT_MODE.upper()}")
    logger.info("=" * 80)
    microbe_to_taxids = load_taxid_hierarchy()
    tasks = []
    for cancer in CANCERS:
        logger.info(f"\nProcessing cancer: {cancer}")
        microbe_list = load_microbe_list(cancer)
        if not microbe_list:
            continue
        for status in STATUSES:
            samples = get_all_samples(cancer, status)
            logger.info(f" {status}: {len(samples)} samples")
            for sample in samples:
                tasks.append((cancer, status, sample, microbe_list, microbe_to_taxids))
                logger.info(f"\nTotal tasks: {len(tasks)}")
                n_processes = min(cpu_count() - 1, 20)
                logger.info(f"Using {n_processes} processes")
                with Pool(n_processes) as pool:
                    all_results = pool.map(process_sample, tasks)
                    all_results = [r for r in all_results if r is not None]
                    logger.info(f"\nProcessed {len(all_results)} samples successfully")
                    logger.info("Summarizing results...")
                    if STAT_MODE == 'cumulative':
                        cancer_summary, global_summary = summarize_results_cumulative(all_results)
                    else:
                        cancer_summary, global_summary = summarize_results_deduplicated(all_results)
                        logger.info("Saving results...")
                        save_results(cancer_summary, global_summary, OUTPUT_DIR)
                        logger.info("Creating visualizations...")
                        create_visualizations(cancer_summary, global_summary, OUTPUT_DIR)
                        logger.info("\n" + "=" * 80)
                        logger.info("Analysis completed successfully!")
                        logger.info(f"Results saved to: {OUTPUT_DIR}")
                        logger.info("=" * 80)


if __name__ == "__main__":
    main()